# **PACR**

In [1]:
# At the START of your notebook, run this cell:

# Copy your PACR code to working directory
import shutil

# If PACR code is uploaded as a dataset
pacr_source = "/kaggle/input/datasets/YOUR_KAGGLE_USERNAME/pacr-detr/pacr-detr.py"  # Update this path
pacr_dest = "/kaggle/working/pacr_detr.py"

shutil.copy(pacr_source, pacr_dest)

# Now you can import it
import sys
sys.path.insert(0, '/kaggle/working')

from pacr_detr import PACRDETRDetector, PACRDETRInference, Config as PACRConfig
print("✓ PACR imported!")

✓ PACR imported!


# **TST**

In [2]:
"""
================================================================================
TST WITH VIDEO SWIN SMALL — FULL TRAINING CODE FOR SAV DATASET
================================================================================
Target: 70%+ mAP on SAV dataset

Key Improvements:
1. Video Swin Small backbone (50M params vs 28M for Tiny)
2. Asymmetric Loss with class reweighting
3. Mixup augmentation for video
4. Cosine annealing with warm restarts
5. Exponential Moving Average (EMA)
6. Gradient checkpointing for memory efficiency
7. Multi-scale temporal sampling

Author: Srinithi R, Divya Maki S, Thirumoorthy K
================================================================================
"""

import os
import sys
import math
import random
import numpy as np
import pandas as pd
from PIL import Image
from collections import defaultdict
from functools import reduce
from operator import mul

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
from torch.utils.checkpoint import checkpoint
from einops import rearrange
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# SEED FOR REPRODUCIBILITY
# ============================================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# ============================================================================
# CONFIGURATION
# ============================================================================
class Config:
    # Paths - UPDATE THESE FOR YOUR KAGGLE ENVIRONMENT
    DATA_ROOT = "/kaggle/input/datasets/YOUR_KAGGLE_USERNAME/sav-dataset/SAV/data/sav"
    FRAMES_DIR = "/kaggle/input/datasets/YOUR_KAGGLE_USERNAME/sav-dataset/SAV/data/sav/frames"
    TRAIN_CSV = "/kaggle/input/datasets/YOUR_KAGGLE_USERNAME/sav-dataset/SAV/annotations/train.csv"
    VAL_CSV = "/kaggle/input/datasets/YOUR_KAGGLE_USERNAME/sav-dataset/SAV/annotations/val.csv"
    OUTPUT_DIR = "/kaggle/working/checkpoints"
    
    # PACR checkpoint (frozen detector)
    PACR_CHECKPOINT = "/kaggle/input/models/YOUR_KAGGLE_USERNAME/pacr-bound/pytorch/default/1/best (1) (1).pth"
    
    # Video Swin Small pretrained weights
    # Download from: https://github.com/SwinTransformer/Video-Swin-Transformer
    SWIN_PRETRAINED = "/kaggle/input/models/YOUR_KAGGLE_USERNAME/swin-small/pytorch/default/1/swin_small_patch244_window877_kinetics400_1k.pth"
    
    # Model architecture
    MODEL_SIZE = 'small'  # 'tiny', 'small', or 'base'
    NUM_CLASSES = 15
    NUM_SEGMENTS = 3      # K=3 temporal segments (TST paper)
    FRAMES_PER_SEGMENT = 16  # T=16 frames per segment
    VIDEO_SIZE = 224
    
    # Video Swin parameters (will be set based on MODEL_SIZE)
    TST_EMBED_DIM = 96
    TST_DEPTHS = (2, 2, 18, 2)  # Small has 18 in stage 3 vs 6 in Tiny
    TST_NUM_HEADS = (3, 6, 12, 24)
    TST_WINDOW_SIZE = (2, 7, 7)  # Temporal window = 2 for K=3 segments
    TST_PATCH_SIZE = (2, 4, 4)
    
    # Training parameters
    BATCH_SIZE = 2
    GRAD_ACCUM_STEPS = 8  # Effective batch = 16
    EPOCHS = 50
    WARMUP_EPOCHS = 3
    
    # Learning rates
    BACKBONE_LR = 5e-5
    HEAD_LR = 5e-4
    MIN_LR = 1e-6
    WEIGHT_DECAY = 0.05
    
    # Augmentation
    USE_MIXUP = True
    MIXUP_ALPHA = 0.8
    CUTMIX_ALPHA = 1.0
    MIXUP_PROB = 0.5
    
    # Loss
    LABEL_SMOOTHING = 0.1
    ASL_GAMMA_NEG = 4
    ASL_GAMMA_POS = 0
    ASL_CLIP = 0.05
    
    # Memory management
    MAX_STUDENTS_PER_BATCH = 8
    USE_GRADIENT_CHECKPOINT = True
    USE_AMP = True
    
    # EMA
    USE_EMA = True
    EMA_DECAY = 0.9998
    
    # Device
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    
    def __init__(self):
        os.makedirs(self.OUTPUT_DIR, exist_ok=True)
        self._set_model_config()
    
    def _set_model_config(self):
        """Set architecture params based on model size"""
        if self.MODEL_SIZE == 'tiny':
            self.TST_EMBED_DIM = 96
            self.TST_DEPTHS = (2, 2, 6, 2)
            self.TST_NUM_HEADS = (3, 6, 12, 24)
        elif self.MODEL_SIZE == 'small':
            self.TST_EMBED_DIM = 96
            self.TST_DEPTHS = (2, 2, 18, 2)
            self.TST_NUM_HEADS = (3, 6, 12, 24)
        elif self.MODEL_SIZE == 'base':
            self.TST_EMBED_DIM = 128
            self.TST_DEPTHS = (2, 2, 18, 2)
            self.TST_NUM_HEADS = (4, 8, 16, 32)

# ============================================================================
# ACTION CLASSES
# ============================================================================
ACTION_NAMES = [
    'sit', 'stand', 'look_forward', 'look_sideways', 'read',
    'flip_books', 'touch_sth', 'raise_hand', 'hands_down', 'take_notes',
    'applaud', 'bend', 'turn_around', 'talk_with_others', 'answer_questions'
]

CLASS_COUNTS = np.array([94513, 6859, 46412, 15163, 39466, 2134, 25951, 
                         5643, 53812, 6550, 2431, 2982, 1525, 3280, 2241])
CLASS_FREQUENCIES = CLASS_COUNTS / CLASS_COUNTS.sum()
RARE_CLASSES = [5, 7, 10, 11, 12, 13, 14]

# ============================================================================
# VIDEO SWIN TRANSFORMER (Supports Tiny/Small/Base)
# ============================================================================
def drop_path(x, drop_prob=0.0, training=False):
    if drop_prob == 0. or not training:
        return x
    keep_prob = 1 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)
    random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
    random_tensor.floor_()
    return x.div(keep_prob) * random_tensor

class DropPath(nn.Module):
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = drop_prob
    def forward(self, x):
        return drop_path(x, self.drop_prob, self.training)

class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

def window_partition(x, window_size):
    B, D, H, W, C = x.shape
    x = x.view(B, D // window_size[0], window_size[0],
               H // window_size[1], window_size[1],
               W // window_size[2], window_size[2], C)
    windows = x.permute(0, 1, 3, 5, 2, 4, 6, 7).contiguous()
    windows = windows.view(-1, reduce(mul, window_size), C)
    return windows

def window_reverse(windows, window_size, B, D, H, W):
    x = windows.view(B, D // window_size[0], H // window_size[1], W // window_size[2],
                     window_size[0], window_size[1], window_size[2], -1)
    x = x.permute(0, 1, 4, 2, 5, 3, 6, 7).contiguous().view(B, D, H, W, -1)
    return x

def get_window_size(x_size, window_size, shift_size=None):
    use_window_size = list(window_size)
    if shift_size is not None:
        use_shift_size = list(shift_size)
    for i in range(len(x_size)):
        if x_size[i] <= window_size[i]:
            use_window_size[i] = x_size[i]
            if shift_size is not None:
                use_shift_size[i] = 0
    if shift_size is None:
        return tuple(use_window_size)
    else:
        return tuple(use_window_size), tuple(use_shift_size)

class WindowAttention3D(nn.Module):
    def __init__(self, dim, window_size, num_heads, qkv_bias=True, qk_scale=None, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** -0.5
        
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size[0] - 1) * (2 * window_size[1] - 1) * (2 * window_size[2] - 1), num_heads))
        
        coords_d = torch.arange(window_size[0])
        coords_h = torch.arange(window_size[1])
        coords_w = torch.arange(window_size[2])
        coords = torch.stack(torch.meshgrid(coords_d, coords_h, coords_w, indexing='ij'))
        coords_flatten = torch.flatten(coords, 1)
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += window_size[0] - 1
        relative_coords[:, :, 1] += window_size[1] - 1
        relative_coords[:, :, 2] += window_size[2] - 1
        relative_coords[:, :, 0] *= (2 * window_size[1] - 1) * (2 * window_size[2] - 1)
        relative_coords[:, :, 1] *= 2 * window_size[2] - 1
        relative_position_index = relative_coords.sum(-1)
        self.register_buffer("relative_position_index", relative_position_index)
        
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        
        nn.init.trunc_normal_(self.relative_position_bias_table, std=.02)
        self.softmax = nn.Softmax(dim=-1)
    
    def forward(self, x, mask=None):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        q = q * self.scale
        attn = q @ k.transpose(-2, -1)
        
        relative_position_bias = self.relative_position_bias_table[
            self.relative_position_index[:N, :N].reshape(-1)].reshape(N, N, -1)
        relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
        attn = attn + relative_position_bias.unsqueeze(0)
        
        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)
        
        attn = self.softmax(attn)
        attn = self.attn_drop(attn)
        
        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

class SwinTransformerBlock3D(nn.Module):
    def __init__(self, dim, num_heads, window_size=(2,7,7), shift_size=(0,0,0),
                 mlp_ratio=4., qkv_bias=True, qk_scale=None, drop=0., attn_drop=0.,
                 drop_path=0., norm_layer=nn.LayerNorm, use_checkpoint=False):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.mlp_ratio = mlp_ratio
        self.use_checkpoint = use_checkpoint
        
        self.norm1 = norm_layer(dim)
        self.attn = WindowAttention3D(
            dim, window_size=window_size, num_heads=num_heads,
            qkv_bias=qkv_bias, qk_scale=qk_scale, attn_drop=attn_drop, proj_drop=drop)
        
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = norm_layer(dim)
        self.mlp = Mlp(in_features=dim, hidden_features=int(dim * mlp_ratio), drop=drop)
    
    def forward_part1(self, x, mask_matrix):
        B, D, H, W, C = x.shape
        window_size, shift_size = get_window_size((D, H, W), self.window_size, self.shift_size)
        
        x = self.norm1(x)
        
        pad_l = pad_t = pad_d0 = 0
        pad_d1 = (window_size[0] - D % window_size[0]) % window_size[0]
        pad_b = (window_size[1] - H % window_size[1]) % window_size[1]
        pad_r = (window_size[2] - W % window_size[2]) % window_size[2]
        x = F.pad(x, (0, 0, pad_l, pad_r, pad_t, pad_b, pad_d0, pad_d1))
        _, Dp, Hp, Wp, _ = x.shape
        
        if any(i > 0 for i in shift_size):
            shifted_x = torch.roll(x, shifts=(-shift_size[0], -shift_size[1], -shift_size[2]), dims=(1, 2, 3))
            attn_mask = mask_matrix
        else:
            shifted_x = x
            attn_mask = None
        
        x_windows = window_partition(shifted_x, window_size)
        attn_windows = self.attn(x_windows, mask=attn_mask)
        
        attn_windows = attn_windows.view(-1, *(window_size + (C,)))
        shifted_x = window_reverse(attn_windows, window_size, B, Dp, Hp, Wp)
        
        if any(i > 0 for i in shift_size):
            x = torch.roll(shifted_x, shifts=(shift_size[0], shift_size[1], shift_size[2]), dims=(1, 2, 3))
        else:
            x = shifted_x
        
        if pad_d1 > 0 or pad_r > 0 or pad_b > 0:
            x = x[:, :D, :H, :W, :].contiguous()
        return x
    
    def forward_part2(self, x):
        return self.drop_path(self.mlp(self.norm2(x)))
    
    def forward(self, x, mask_matrix):
        shortcut = x
        if self.use_checkpoint:
            x = checkpoint(self.forward_part1, x, mask_matrix, use_reentrant=False)
        else:
            x = self.forward_part1(x, mask_matrix)
        x = shortcut + self.drop_path(x)
        
        if self.use_checkpoint:
            x = x + checkpoint(self.forward_part2, x, use_reentrant=False)
        else:
            x = x + self.forward_part2(x)
        return x

class PatchMerging(nn.Module):
    def __init__(self, dim, norm_layer=nn.LayerNorm):
        super().__init__()
        self.dim = dim
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)
        self.norm = norm_layer(4 * dim)
    
    def forward(self, x):
        B, D, H, W, C = x.shape
        
        pad_h = (2 - H % 2) % 2
        pad_w = (2 - W % 2) % 2
        if pad_h > 0 or pad_w > 0:
            x = F.pad(x, (0, 0, 0, pad_w, 0, pad_h))
        
        _, _, H, W, _ = x.shape
        
        x0 = x[:, :, 0::2, 0::2, :]
        x1 = x[:, :, 1::2, 0::2, :]
        x2 = x[:, :, 0::2, 1::2, :]
        x3 = x[:, :, 1::2, 1::2, :]
        x = torch.cat([x0, x1, x2, x3], -1)
        
        x = self.norm(x)
        x = self.reduction(x)
        return x

class PatchEmbed3D(nn.Module):
    def __init__(self, patch_size=(2,4,4), in_chans=3, embed_dim=96, norm_layer=None):
        super().__init__()
        self.patch_size = patch_size
        self.in_chans = in_chans
        self.embed_dim = embed_dim
        self.proj = nn.Conv3d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = norm_layer(embed_dim) if norm_layer else None
    
    def forward(self, x):
        _, _, D, H, W = x.shape
        if W % self.patch_size[2] != 0:
            x = F.pad(x, (0, self.patch_size[2] - W % self.patch_size[2]))
        if H % self.patch_size[1] != 0:
            x = F.pad(x, (0, 0, 0, self.patch_size[1] - H % self.patch_size[1]))
        if D % self.patch_size[0] != 0:
            x = F.pad(x, (0, 0, 0, 0, 0, self.patch_size[0] - D % self.patch_size[0]))
        
        x = self.proj(x)
        if self.norm:
            D, Wh, Ww = x.size(2), x.size(3), x.size(4)
            x = x.flatten(2).transpose(1, 2)
            x = self.norm(x)
            x = x.transpose(1, 2).view(-1, self.embed_dim, D, Wh, Ww)
        return x

class BasicLayer(nn.Module):
    def __init__(self, dim, depth, num_heads, window_size=(2,7,7),
                 mlp_ratio=4., qkv_bias=True, qk_scale=None, drop=0., attn_drop=0.,
                 drop_path=0., norm_layer=nn.LayerNorm, downsample=None, use_checkpoint=False):
        super().__init__()
        self.window_size = window_size
        self.shift_size = tuple(i // 2 for i in window_size)
        self.depth = depth
        self.use_checkpoint = use_checkpoint
        
        self.blocks = nn.ModuleList([
            SwinTransformerBlock3D(
                dim=dim, num_heads=num_heads, window_size=window_size,
                shift_size=(0,0,0) if (i % 2 == 0) else self.shift_size,
                mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, qk_scale=qk_scale,
                drop=drop, attn_drop=attn_drop,
                drop_path=drop_path[i] if isinstance(drop_path, list) else drop_path,
                norm_layer=norm_layer, use_checkpoint=use_checkpoint)
            for i in range(depth)])
        
        self.downsample = downsample
    
    def forward(self, x):
        B, C, D, H, W = x.shape
        window_size, shift_size = get_window_size((D, H, W), self.window_size, self.shift_size)
        
        x = x.permute(0, 2, 3, 4, 1)  # B, D, H, W, C
        
        Dp = int(np.ceil(D / window_size[0])) * window_size[0]
        Hp = int(np.ceil(H / window_size[1])) * window_size[1]
        Wp = int(np.ceil(W / window_size[2])) * window_size[2]
        attn_mask = self.compute_mask(Dp, Hp, Wp, window_size, shift_size, x.device)
        
        for blk in self.blocks:
            x = blk(x, attn_mask)
        
        x = x.permute(0, 4, 1, 2, 3)  # B, C, D, H, W
        
        if self.downsample is not None:
            x = x.permute(0, 2, 3, 4, 1)  # B, D, H, W, C
            x = self.downsample(x)
            x = x.permute(0, 4, 1, 2, 3)  # B, C, D, H, W
        return x
    
    def compute_mask(self, D, H, W, window_size, shift_size, device):
        if any(s > 0 for s in shift_size):
            img_mask = torch.zeros((1, D, H, W, 1), device=device)
            cnt = 0
            for d in (slice(0, -window_size[0]), slice(-window_size[0], -shift_size[0]), slice(-shift_size[0], None)):
                for h in (slice(0, -window_size[1]), slice(-window_size[1], -shift_size[1]), slice(-shift_size[1], None)):
                    for w in (slice(0, -window_size[2]), slice(-window_size[2], -shift_size[2]), slice(-shift_size[2], None)):
                        img_mask[:, d, h, w, :] = cnt
                        cnt += 1
            
            mask_windows = window_partition(img_mask, window_size)
            mask_windows = mask_windows.squeeze(-1)
            attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
            attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(attn_mask == 0, float(0.0))
        else:
            attn_mask = None
        return attn_mask

class VideoSwinTransformer(nn.Module):
    """Video Swin Transformer supporting Tiny/Small/Base variants"""
    
    def __init__(self, patch_size=(2,4,4), in_chans=3, embed_dim=96,
                 depths=(2,2,6,2), num_heads=(3,6,12,24), window_size=(2,7,7),
                 mlp_ratio=4., qkv_bias=True, qk_scale=None, drop_rate=0.,
                 attn_drop_rate=0., drop_path_rate=0.2, norm_layer=nn.LayerNorm,
                 patch_norm=False, use_checkpoint=False):
        super().__init__()
        
        self.num_layers = len(depths)
        self.embed_dim = embed_dim
        self.patch_norm = patch_norm
        self.num_features = int(embed_dim * 2 ** (self.num_layers - 1))
        self.mlp_ratio = mlp_ratio
        
        self.patch_embed = PatchEmbed3D(
            patch_size=patch_size, in_chans=in_chans, embed_dim=embed_dim,
            norm_layer=norm_layer if patch_norm else None)
        
        self.pos_drop = nn.Dropout(p=drop_rate)
        
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]
        
        self.layers = nn.ModuleList()
        for i_layer in range(self.num_layers):
            layer = BasicLayer(
                dim=int(embed_dim * 2 ** i_layer),
                depth=depths[i_layer],
                num_heads=num_heads[i_layer],
                window_size=window_size,
                mlp_ratio=mlp_ratio,
                qkv_bias=qkv_bias,
                qk_scale=qk_scale,
                drop=drop_rate,
                attn_drop=attn_drop_rate,
                drop_path=dpr[sum(depths[:i_layer]):sum(depths[:i_layer + 1])],
                norm_layer=norm_layer,
                downsample=PatchMerging(int(embed_dim * 2 ** i_layer)) if i_layer < self.num_layers - 1 else None,
                use_checkpoint=use_checkpoint)
            self.layers.append(layer)
        
        self.norm = norm_layer(self.num_features)
        self.avgpool = nn.AdaptiveAvgPool3d(1)
        
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
    
    def forward(self, x):
        x = self.patch_embed(x)
        x = self.pos_drop(x)
        
        for layer in self.layers:
            x = layer(x)
        
        x = x.permute(0, 2, 3, 4, 1)  # B, D, H, W, C
        x = self.norm(x)
        x = x.permute(0, 4, 1, 2, 3)  # B, C, D, H, W
        x = self.avgpool(x)
        x = x.flatten(1)
        
        return x

# ============================================================================
# COSINE CLASSIFIER WITH LEARNABLE TEMPERATURE
# ============================================================================
class CosineClassifier(nn.Module):
    def __init__(self, in_features, num_classes, scale=16.0):
        super().__init__()
        self.in_features = in_features
        self.num_classes = num_classes
        self.scale = scale
        self.weight = nn.Parameter(torch.Tensor(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
    
    def forward(self, x):
        x_norm = F.normalize(x, dim=-1)
        w_norm = F.normalize(self.weight, dim=-1)
        return self.scale * F.linear(x_norm, w_norm)

# ============================================================================
# TST (TEMPORAL SEGMENT TRANSFORMER)
# ============================================================================
class TST(nn.Module):
    """Temporal Segment Transformer with Video Swin backbone"""
    
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.num_segments = config.NUM_SEGMENTS
        
        # Video Swin backbone
        self.video_swin = VideoSwinTransformer(
            patch_size=config.TST_PATCH_SIZE,
            embed_dim=config.TST_EMBED_DIM,
            depths=config.TST_DEPTHS,
            num_heads=config.TST_NUM_HEADS,
            window_size=config.TST_WINDOW_SIZE,
            drop_path_rate=0.2,
            use_checkpoint=config.USE_GRADIENT_CHECKPOINT
        )
        
        # Feature dimension from backbone
        self.num_features = self.video_swin.num_features  # 768 for Small
        
        # Feature projection
        self.feature_proj = nn.Sequential(
            nn.Linear(self.num_features, 512),
            nn.LayerNorm(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )
        
        # Cosine classifier
        self.classifier = CosineClassifier(256, config.NUM_CLASSES, scale=16.0)
        
        # Per-class learnable temperature
        self.class_temp = nn.Parameter(torch.ones(config.NUM_CLASSES))
    
    def forward(self, tubes):
        """
        Args:
            tubes: (N, K*T, C, H, W) where K=num_segments, T=frames_per_segment
        Returns:
            logits: (N, num_classes)
        """
        N = tubes.size(0)
        K = self.num_segments
        T = self.config.FRAMES_PER_SEGMENT
        
        # Reshape: (N, K*T, C, H, W) -> (N*K, C, T, H, W)
        tubes = tubes.view(N, K, T, 3, 224, 224)
        tubes = tubes.permute(0, 1, 3, 2, 4, 5)  # (N, K, C, T, H, W)
        tubes = tubes.reshape(N * K, 3, T, 224, 224)
        
        # Extract features from each segment
        features = self.video_swin(tubes)  # (N*K, num_features)
        
        # Aggregate segments (average pooling)
        features = features.view(N, K, -1)  # (N, K, num_features)
        features = features.mean(dim=1)  # (N, num_features)
        
        # Project and classify
        features = self.feature_proj(features)  # (N, 256)
        logits = self.classifier(features)  # (N, num_classes)
        
        # Apply per-class temperature
        logits = logits / (self.class_temp.unsqueeze(0) + 1e-6)
        
        return logits

# ============================================================================
# ASYMMETRIC LOSS WITH CLASS REWEIGHTING
# ============================================================================
class AsymmetricLossOptimized(nn.Module):
    """ASL with rare class boosting and label smoothing"""
    
    def __init__(self, gamma_neg=4, gamma_pos=0, clip=0.05, 
                 label_smoothing=0.1, class_weights=None):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.label_smoothing = label_smoothing
        
        # Class weights based on inverse frequency + rare class boost
        if class_weights is None:
            weights = 1.0 / (CLASS_FREQUENCIES + 0.01)
            weights = weights / weights.mean()
            
            # Extra boost for weak classes
            boost = np.ones(15)
            boost[13] = 5.0  # talk_with_others
            boost[12] = 3.0  # turn_around
            boost[3] = 2.0   # look_sideways
            boost[5] = 2.0   # flip_books
            boost[14] = 2.0  # answer_questions
            
            weights = weights * boost
            self.register_buffer('class_weights', torch.tensor(weights, dtype=torch.float32))
        else:
            self.register_buffer('class_weights', class_weights)
    
    def forward(self, outputs):
        logits = outputs['logits']
        labels = outputs['labels']
        
        if logits is None or labels is None:
            return torch.tensor(0.0, requires_grad=True)
        
        labels = labels.to(logits.device).float()
        
        # Label smoothing
        if self.label_smoothing > 0:
            labels = labels * (1 - self.label_smoothing) + self.label_smoothing / 2
        
        # Asymmetric loss computation
        x_sig = torch.sigmoid(logits)
        xs_pos = x_sig
        xs_neg = 1 - x_sig
        
        # Asymmetric clipping
        if self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1.0)
        
        # Binary cross entropy
        los_pos = labels * torch.log(xs_pos.clamp(min=1e-8))
        los_neg = (1 - labels) * torch.log(xs_neg.clamp(min=1e-8))
        loss = los_pos + los_neg
        
        # Asymmetric focusing
        if self.gamma_neg > 0 or self.gamma_pos > 0:
            pt = xs_pos * labels + xs_neg * (1 - labels)
            gamma = self.gamma_pos * labels + self.gamma_neg * (1 - labels)
            loss *= (1 - pt) ** gamma
        
        # Class weights
        loss = loss * self.class_weights.unsqueeze(0)
        
        return -loss.mean()

# ============================================================================
# MIXUP AUGMENTATION FOR VIDEO
# ============================================================================
class VideoMixup:
    """Mixup and CutMix augmentation for video data"""
    
    def __init__(self, mixup_alpha=0.8, cutmix_alpha=1.0, prob=0.5):
        self.mixup_alpha = mixup_alpha
        self.cutmix_alpha = cutmix_alpha
        self.prob = prob
    
    def __call__(self, tubes, labels):
        if random.random() > self.prob:
            return tubes, labels, labels, 1.0
        
        batch_size = tubes.size(0)
        if batch_size < 2:
            return tubes, labels, labels, 1.0
        
        # Random permutation
        indices = torch.randperm(batch_size, device=tubes.device)
        
        # Choose mixup or cutmix
        if random.random() < 0.5 and self.mixup_alpha > 0:
            # Mixup
            lam = np.random.beta(self.mixup_alpha, self.mixup_alpha)
            tubes = lam * tubes + (1 - lam) * tubes[indices]
        else:
            # CutMix (temporal)
            lam = np.random.beta(self.cutmix_alpha, self.cutmix_alpha)
            T = tubes.size(1)
            cut_len = int(T * (1 - lam))
            cut_start = random.randint(0, T - cut_len)
            tubes[:, cut_start:cut_start+cut_len] = tubes[indices, cut_start:cut_start+cut_len]
            lam = 1 - cut_len / T
        
        return tubes, labels, labels[indices], lam

# ============================================================================
# EXPONENTIAL MOVING AVERAGE
# ============================================================================
class EMA:
    """Exponential Moving Average for model weights"""
    
    def __init__(self, model, decay=0.9998):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()
    
    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_average.clone()
    
    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data
                param.data = self.shadow[name]
    
    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]
        self.backup = {}

# ============================================================================
# FIXED SAV DATASET — MATCHES YOUR ACTUAL CSV FORMAT
# ============================================================================

class SAVDataset(Dataset):
    """SAV Dataset — CSV has NO HEADERS"""
    
    def __init__(self, data_root, csv_path, config, train=True):
        self.data_root = data_root
        self.frames_dir = config.FRAMES_DIR
        self.config = config
        self.train = train
        self.num_segments = config.NUM_SEGMENTS
        self.frames_per_segment = config.FRAMES_PER_SEGMENT
        self.spatial_size = config.VIDEO_SIZE
        
        # ═══ CRITICAL: SAV CSV has NO headers ═══
        # Format: video_clip_id, constant, x1, y1, x2, y2, action_id, person_id
        df = pd.read_csv(csv_path, header=None,
            names=['video_clip_id', 'constant', 'x1', 'y1', 'x2', 'y2', 'action_id', 'person_id'])
        
        # Group by video and person
        self.video_clips = self._group_annotations(df)
        self.video_ids = list(self.video_clips.keys())
        
        # Compute sample weights for rare class oversampling
        self.sample_weights = self._compute_sample_weights()
        
        print(f"{'Train' if train else 'Val'} dataset: {len(self.video_ids)} clips")
    
    def _group_annotations(self, df):
        """Group annotations by video, then by person"""
        grouped = defaultdict(lambda: defaultdict(list))
        
        for _, row in df.iterrows():
            vid = str(row['video_clip_id'])
            try:
                pid = int(row['person_id'])
                # Action IDs are 1-indexed in CSV, convert to 0-indexed
                action_id = int(row['action_id']) - 1
                if 0 <= action_id < 15:
                    grouped[vid][pid].append({
                        'bbox': [float(row['x1']), float(row['y1']),
                                 float(row['x2']), float(row['y2'])],
                        'action_id': action_id
                    })
            except:
                continue
        
        # Aggregate per video: each student has bbox + list of actions
        video_clips = {}
        for vid, persons in grouped.items():
            students = []
            for pid, annos in persons.items():
                # Use middle annotation's bbox (they're all the same keyframe)
                mid_idx = len(annos) // 2
                bbox = annos[mid_idx]['bbox']
                # Collect all unique actions for this person
                actions = list(set([a['action_id'] for a in annos]))
                students.append({
                    'person_id': pid,
                    'bbox': bbox,
                    'actions': actions
                })
            if students:
                video_clips[vid] = students
        
        return video_clips
    
    def _compute_sample_weights(self):
        """Compute sampling weights — oversample rare classes"""
        WEAK_CLASSES = {13, 12, 3, 5, 7, 14}  # talk_with_others, turn_around, etc.
        
        weights = []
        for vid in self.video_ids:
            actions = set()
            for s in self.video_clips[vid]:
                for a in s['actions']:
                    actions.add(a)
            
            # Higher weight for clips with weak classes
            if actions & WEAK_CLASSES:
                w = 5.0 if 13 in actions else (4.0 if 12 in actions else 3.0)
            else:
                w = 1.0
            weights.append(w)
        
        return weights
    
    def __len__(self):
        return len(self.video_ids)
    
    def _load_frames(self, vid):
        """Load K*T frames for temporal input"""
        clip_dir = os.path.join(self.frames_dir, str(vid))
        if not os.path.isdir(clip_dir):
            return None
        
        frame_files = sorted([f for f in os.listdir(clip_dir) 
                              if f.endswith(('.jpg', '.png'))])
        
        K = self.num_segments
        T = self.frames_per_segment
        total_needed = K * T
        
        if len(frame_files) < total_needed:
            return None
        
        # Sample frames: K segments, T frames each
        total = len(frame_files)
        seg_dur = total / K
        
        frames = []
        for s in range(K):
            seg_start = int(s * seg_dur)
            seg_end = int((s + 1) * seg_dur)
            idxs = np.linspace(seg_start, max(seg_start, seg_end - 1), T, dtype=int)
            
            for idx in idxs:
                idx = min(int(idx), total - 1)
                frame_path = os.path.join(clip_dir, frame_files[idx])
                try:
                    img = Image.open(frame_path).convert('RGB')
                    img = img.resize((self.spatial_size, self.spatial_size), Image.BILINEAR)
                    img = np.array(img, dtype=np.float32) / 255.0
                    # ImageNet normalization
                    img = (img - np.array([0.485, 0.456, 0.406])) / np.array([0.229, 0.224, 0.225])
                    frames.append(img)
                except:
                    frames.append(np.zeros((self.spatial_size, self.spatial_size, 3), dtype=np.float32))
        
        # (K*T, H, W, C) -> (K*T, C, H, W)
        frames = np.stack(frames, axis=0)
        frames = np.transpose(frames, (0, 3, 1, 2))
        return torch.tensor(frames, dtype=torch.float32)
    
    def _prepare_spatial(self, vid):
        """Load middle frame for spatial/detection input"""
        clip_dir = os.path.join(self.frames_dir, str(vid))
        if not os.path.isdir(clip_dir):
            return torch.zeros(3, self.spatial_size, self.spatial_size)
        
        frame_files = sorted([f for f in os.listdir(clip_dir) 
                              if f.endswith(('.jpg', '.png'))])
        if not frame_files:
            return torch.zeros(3, self.spatial_size, self.spatial_size)
        
        # Middle frame (keyframe annotation)
        mid_idx = len(frame_files) // 2
        frame_path = os.path.join(clip_dir, frame_files[mid_idx])
        
        try:
            img = Image.open(frame_path).convert('RGB')
            img = img.resize((self.spatial_size, self.spatial_size), Image.BILINEAR)
            img = np.array(img, dtype=np.float32) / 255.0
            img = (img - np.array([0.485, 0.456, 0.406])) / np.array([0.229, 0.224, 0.225])
            img = np.transpose(img, (2, 0, 1))
            return torch.tensor(img, dtype=torch.float32)
        except:
            return torch.zeros(3, self.spatial_size, self.spatial_size)
    
    def _prepare_targets(self, students):
        """Prepare GT boxes and multi-label targets"""
        # SAV coordinates are RELATIVE [0,1], scale to spatial_size
        H = W = self.spatial_size
        boxes = []
        labels = []
        
        for s in students:
            x1, y1, x2, y2 = s['bbox']
            # Scale relative coords to pixel coords
            boxes.append([x1 * W, y1 * H, x2 * W, y2 * H])
            
            # Multi-label: one-hot for all actions
            label = torch.zeros(15, dtype=torch.float32)
            for a in s['actions']:
                if 0 <= a < 15:
                    label[a] = 1.0
            labels.append(label)
        
        return {
            'boxes': torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4)),
            'labels': torch.stack(labels) if labels else torch.zeros((0, 15))
        }
    
    def __getitem__(self, idx):
        vid = self.video_ids[idx]
        students = self.video_clips[vid]
        
        # Spatial frame (for PACR detection, though we use GT boxes)
        spatial = self._prepare_spatial(vid)
        
        # Temporal frames (for TST classification)
        temporal = self._load_frames(vid)
        if temporal is None:
            K, T = self.num_segments, self.frames_per_segment
            temporal = torch.zeros(K * T, 3, self.spatial_size, self.spatial_size)
        
        # Targets (GT boxes and labels)
        targets = self._prepare_targets(students)
        targets['video_id'] = vid
        
        return spatial, temporal, targets


def collate_fn(batch):
    """Collate function for DataLoader"""
    spatial = torch.stack([item[0] for item in batch])
    temporal = torch.stack([item[1] for item in batch])
    targets = [item[2] for item in batch]
    return spatial, temporal, targets
# ============================================================================
# PACR-DETR DETECTOR (Frozen)
# ============================================================================
# Import PACR from your existing code
# sys.path.insert(0, '/path/to/pacr')
# from pacr_detr import PACRDETRDetector, PACRConfig

# For this code, we assume PACR is already imported and available
# If not, you need to include the PACR code here

class PACRWithTST(nn.Module):
    """Combined PACR detection + TST classification"""
    
    def __init__(self, config, use_real_pacr=True):
        super().__init__()
        self.config = config
        
        # Use your actual PACR detector
        if use_real_pacr:
            pacr_config = PACRConfig()
            self.detector = PACRDETRDetector(pacr_config)
        else:
            self.detector = nn.Identity()  # Placeholder
        
        # TST classifier
        self.tst = TST(config)
    
    def freeze_pacr(self):
        """Freeze PACR detector"""
        for param in self.detector.parameters():
            param.requires_grad = False
        self.detector.eval()
        print("✓ PACR frozen")
    
    def load_pacr_checkpoint(self, path):
        """Load PACR checkpoint"""
        if not os.path.exists(path):
            print(f"⚠ PACR checkpoint not found: {path}")
            return False
        
        ckpt = torch.load(path, map_location='cpu', weights_only=False)
        state = ckpt.get('model_state_dict', ckpt)
        state = {k.replace('module.', ''): v for k, v in state.items()}
        
        # Load with shape checking
        model_state = self.detector.state_dict()
        loaded = 0
        for k, v in state.items():
            if k in model_state and v.shape == model_state[k].shape:
                model_state[k] = v
                loaded += 1
        
        self.detector.load_state_dict(model_state)
        print(f"✓ PACR checkpoint loaded: {loaded}/{len(model_state)} keys")
        return True
    
    def forward(self, spatial, temporal, targets=None):
        """
        Args:
            spatial: (B, C, H, W) - spatial frame for detection
            temporal: (B, K*T, C, H, W) - temporal frames for TST
            targets: list of dicts, each with 'boxes' (N,4) and 'labels' (N,15)
        """
        B = spatial.size(0)
        device = spatial.device
        
        all_logits = []
        all_labels = []
        student_counts = []
        
        for b in range(B):
            # ═══ FIXED: targets[b] is a SINGLE dict ═══
            if targets is not None and targets[b]['boxes'].size(0) > 0:
                boxes = targets[b]['boxes'].to(device)
                labels = targets[b]['labels'].to(device)
            else:
                # Inference: use PACR detection
                with torch.no_grad():
                    det_out = self.detector(spatial[b:b+1])
                    boxes = det_out.get('pred_boxes', None)
                    if boxes is None or boxes.size(0) == 0:
                        student_counts.append(0)
                        continue
                labels = None
            
            num_students = boxes.size(0)
            student_counts.append(num_students)
            
            if num_students == 0:
                continue
            
            # Cap students for memory
            if num_students > self.config.MAX_STUDENTS_PER_BATCH:
                indices = torch.randperm(num_students)[:self.config.MAX_STUDENTS_PER_BATCH]
                boxes = boxes[indices]
                if labels is not None:
                    labels = labels[indices]
                num_students = self.config.MAX_STUDENTS_PER_BATCH
            
            # Extract tubes and classify
            tubes = self._extract_tubes(temporal[b], boxes)
            logits = self.tst(tubes)
            
            all_logits.append(logits)
            if labels is not None:
                all_labels.append(labels)
        
        if all_logits:
            all_logits = torch.cat(all_logits, dim=0)
            all_labels = torch.cat(all_labels, dim=0) if all_labels else None
        else:
            all_logits = None
            all_labels = None
        
        return {
            'logits': all_logits,
            'labels': all_labels,
            'student_counts': student_counts
        }
    
    def _extract_tubes(self, temporal, boxes):
        """Extract tubes - boxes are in PIXEL coordinates"""
        K_T, C, H, W = temporal.shape
        N = boxes.size(0)
        
        tubes = []
        for i in range(N):
            x1, y1, x2, y2 = boxes[i].int().tolist()
            
            # Clamp
            x1 = max(0, min(x1, W - 1))
            y1 = max(0, min(y1, H - 1))
            x2 = max(x1 + 1, min(x2, W))
            y2 = max(y1 + 1, min(y2, H))
            
            student_tube = []
            for t in range(K_T):
                crop = temporal[t, :, y1:y2, x1:x2]
                crop = F.interpolate(crop.unsqueeze(0), size=(224, 224), 
                                    mode='bilinear', align_corners=False)
                student_tube.append(crop.squeeze(0))
            
            tubes.append(torch.stack(student_tube, dim=0))
        
        return torch.stack(tubes, dim=0)

# ============================================================================
# METRICS
# ============================================================================
def compute_metrics(logits, labels):
    """Compute mAP, F1, precision, recall"""
    preds = torch.sigmoid(logits).numpy()
    labels = labels.numpy()
    
    aps = []
    for c in range(15):
        if labels[:, c].sum() == 0:
            aps.append(0.0)
            continue
        
        # Sort by prediction score
        idx = np.argsort(-preds[:, c])
        tp = np.cumsum(labels[idx, c])
        fp = np.cumsum(1 - labels[idx, c])
        prec = tp / (tp + fp + 1e-6)
        rec = tp / (labels[:, c].sum() + 1e-6)
        
        # 11-point interpolated AP
        ap = 0.0
        for t in np.arange(0, 1.1, 0.1):
            p = prec[rec >= t].max() if len(prec[rec >= t]) > 0 else 0
            ap += p
        aps.append(ap / 11)
    
    mAP = np.mean(aps)
    
    # F1, precision, recall
    thresholds = np.where(CLASS_FREQUENCIES < 0.02, 0.3, 0.5)
    pred_binary = (preds > thresholds).astype(int)
    
    tp = (pred_binary * labels).sum()
    fp = (pred_binary * (1 - labels)).sum()
    fn = ((1 - pred_binary) * labels).sum()
    
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    f1 = 2 * precision * recall / (precision + recall + 1e-6)
    
    return {
        'mAP': mAP,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'aps': aps
    }

# ============================================================================
# LOAD PRETRAINED WEIGHTS WITH WINDOW SIZE INTERPOLATION
# ============================================================================
def load_swin_pretrained(model, pretrained_path, target_window_size=(2, 7, 7)):
    """Load Kinetics-400 pretrained weights with window size interpolation"""
    
    if not os.path.exists(pretrained_path):
        print(f"⚠ Pretrained weights not found: {pretrained_path}")
        return False
    
    checkpoint = torch.load(pretrained_path, map_location='cpu', weights_only=False)
    state_dict = checkpoint.get('model', checkpoint.get('state_dict', checkpoint))
    
    # Remove 'backbone.' prefix if present
    state_dict = {k.replace('backbone.', ''): v for k, v in state_dict.items()}
    
    # Get model state dict
    model_dict = model.state_dict()
    
    # Interpolate relative_position_bias_table for window size mismatch
    source_window = (8, 7, 7)  # Kinetics pretrained window size
    
    new_state_dict = {}
    for k, v in state_dict.items():
        if k in model_dict:
            if 'relative_position_bias_table' in k and v.shape != model_dict[k].shape:
                # Interpolate
                print(f"  Interpolating {k}: {v.shape} -> {model_dict[k].shape}")
                
                # Original dimensions
                src_size = int(((v.shape[0]) ** 0.5))
                # This is more complex for 3D, need proper handling
                
                # For 3D, reshape and interpolate
                num_heads = v.shape[1]
                
                # Source: (2*Wd-1) * (2*Wh-1) * (2*Ww-1) x num_heads
                src_d, src_h, src_w = 2*source_window[0]-1, 2*source_window[1]-1, 2*source_window[2]-1
                tgt_d, tgt_h, tgt_w = 2*target_window_size[0]-1, 2*target_window_size[1]-1, 2*target_window_size[2]-1
                
                # Reshape to 3D
                v_3d = v.reshape(src_d, src_h, src_w, num_heads).permute(3, 0, 1, 2)  # (heads, D, H, W)
                
                # Interpolate
                v_3d = F.interpolate(v_3d.unsqueeze(0), size=(tgt_d, tgt_h, tgt_w), 
                                    mode='trilinear', align_corners=False).squeeze(0)
                
                # Reshape back
                v = v_3d.permute(1, 2, 3, 0).reshape(-1, num_heads)
                
            if v.shape == model_dict[k].shape:
                new_state_dict[k] = v
            else:
                print(f"  Skipping {k}: shape mismatch {v.shape} vs {model_dict[k].shape}")
        else:
            pass  # Skip keys not in model
    
    # Load
    model.load_state_dict(new_state_dict, strict=False)
    
    loaded = len(new_state_dict)
    total = len(model_dict)
    print(f"✓ Loaded {loaded}/{total} keys from pretrained weights")
    
    return True

# ============================================================================
# TRAINING FUNCTION
# ============================================================================
# ============================================================================
# TRAINING FUNCTION — WITH PROPER CHECKPOINT SAVING
# ============================================================================
def train(config, resume_path=None):
    """Main training loop with resume support"""
    
    device = torch.device(config.DEVICE)
    
    print("=" * 70)
    print("TST WITH VIDEO SWIN SMALL — TRAINING")
    print("=" * 70)
    print(f"Model: Video Swin {config.MODEL_SIZE.upper()}")
    print(f"Segments: K={config.NUM_SEGMENTS}, T={config.FRAMES_PER_SEGMENT}")
    print(f"Window size: {config.TST_WINDOW_SIZE}")
    print(f"Batch: {config.BATCH_SIZE} x {config.GRAD_ACCUM_STEPS} = {config.BATCH_SIZE * config.GRAD_ACCUM_STEPS}")
    print(f"LR: backbone={config.BACKBONE_LR}, head={config.HEAD_LR}")
    print("=" * 70)
    
    # ══════════════════════════════════════════════════════════════════════
    # BUILD MODEL
    # ══════════════════════════════════════════════════════════════════════
    pacr_config = PACRConfig()
    model = PACRWithTST(config, pacr_config)
    
    # Load PACR checkpoint
    if os.path.exists(config.PACR_CHECKPOINT):
        model.load_pacr_checkpoint(config.PACR_CHECKPOINT)
    model.freeze_pacr()
    
    # Load Swin pretrained (only if NOT resuming)
    if resume_path is None and os.path.exists(config.SWIN_PRETRAINED):
        print(f"\nLoading pretrained weights: {config.SWIN_PRETRAINED}")
        load_swin_pretrained(model.tst.video_swin, config.SWIN_PRETRAINED, config.TST_WINDOW_SIZE)
    
    model = model.to(device)
    
    # Multi-GPU
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs")
        model.tst = nn.DataParallel(model.tst)
    
    tst_module = model.tst.module if isinstance(model.tst, nn.DataParallel) else model.tst
    
    # ══════════════════════════════════════════════════════════════════════
    # DATASETS
    # ══════════════════════════════════════════════════════════════════════
    print("\nCreating datasets...")
    train_ds = SAVDataset(config.DATA_ROOT, config.TRAIN_CSV, config, train=True)
    val_ds = SAVDataset(config.DATA_ROOT, config.VAL_CSV, config, train=False)
    
    sampler = WeightedRandomSampler(train_ds.sample_weights, len(train_ds), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, sampler=sampler,
                              num_workers=4, collate_fn=collate_fn, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=config.BATCH_SIZE, shuffle=False,
                            num_workers=4, collate_fn=collate_fn, pin_memory=True)
    
    # ══════════════════════════════════════════════════════════════════════
    # OPTIMIZER & SCHEDULER
    # ══════════════════════════════════════════════════════════════════════
    backbone_params = list(tst_module.video_swin.parameters())
    head_params = (list(tst_module.feature_proj.parameters()) +
                   list(tst_module.classifier.parameters()) +
                   [tst_module.class_temp])
    
    print(f"\nBackbone params: {sum(p.numel() for p in backbone_params):,}")
    print(f"Head params: {sum(p.numel() for p in head_params):,}")
    
    optimizer = optim.AdamW([
        {'params': backbone_params, 'lr': config.BACKBONE_LR},
        {'params': head_params, 'lr': config.HEAD_LR},
    ], weight_decay=config.WEIGHT_DECAY)
    
    steps_per_epoch = len(train_loader) // config.GRAD_ACCUM_STEPS
    warmup_steps = config.WARMUP_EPOCHS * steps_per_epoch
    total_steps = config.EPOCHS * steps_per_epoch
    
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        else:
            progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
            return config.MIN_LR / config.HEAD_LR + (1 - config.MIN_LR / config.HEAD_LR) * 0.5 * (1 + math.cos(math.pi * progress))
    
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    
    # Loss & AMP
    criterion = AsymmetricLossOptimized(
        gamma_neg=config.ASL_GAMMA_NEG,
        gamma_pos=config.ASL_GAMMA_POS,
        clip=config.ASL_CLIP,
        label_smoothing=config.LABEL_SMOOTHING
    ).to(device)
    
    scaler = GradScaler() if config.USE_AMP else None
    
    # EMA
    ema = None
    if config.USE_EMA:
        ema = EMA(tst_module, decay=config.EMA_DECAY)
    
    # ══════════════════════════════════════════════════════════════════════
    # RESUME FROM CHECKPOINT
    # ══════════════════════════════════════════════════════════════════════
    start_epoch = 1
    best_mAP = 0.0
    global_step = 0
    
    if resume_path is not None and os.path.exists(resume_path):
        print(f"\n{'='*70}")
        print(f"RESUMING FROM: {resume_path}")
        print(f"{'='*70}")
        
        checkpoint = torch.load(resume_path, map_location=device, weights_only=False)
        
        # Load model weights
        tst_state = checkpoint['tst_state_dict']
        tst_state = {k.replace('module.', ''): v for k, v in tst_state.items()}
        tst_module.load_state_dict(tst_state)
        print(f"  ✓ Model weights loaded")
        
        # Load optimizer state
        if 'optimizer_state_dict' in checkpoint:
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            print(f"  ✓ Optimizer state loaded")
        
        # Load scheduler state
        if 'scheduler_state_dict' in checkpoint:
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            print(f"  ✓ Scheduler state loaded")
        
        # Load scaler state
        if 'scaler_state_dict' in checkpoint and scaler is not None:
            scaler.load_state_dict(checkpoint['scaler_state_dict'])
            print(f"  ✓ Scaler state loaded")
        
        # Load EMA state
        if 'ema_shadow' in checkpoint and ema is not None:
            ema.shadow = checkpoint['ema_shadow']
            print(f"  ✓ EMA state loaded")
        
        # Load training state
        start_epoch = checkpoint.get('epoch', 0) + 1
        best_mAP = checkpoint.get('best_mAP', 0.0)
        global_step = checkpoint.get('global_step', 0)
        
        print(f"  ✓ Resuming from epoch {start_epoch}, best mAP: {best_mAP:.4f}")
        print(f"  ✓ Global step: {global_step}")
        print(f"  ✓ Current LR: backbone={scheduler.get_last_lr()[0]:.2e}, head={scheduler.get_last_lr()[1]:.2e}")
        print(f"{'='*70}\n")
    
    # ══════════════════════════════════════════════════════════════════════
    # TRAINING LOOP
    # ══════════════════════════════════════════════════════════════════════
    for epoch in range(start_epoch, config.EPOCHS + 1):
        print(f"\nEpoch {epoch}/{config.EPOCHS}")
        
        model.tst.train()
        model.detector.eval()
        optimizer.zero_grad()
        
        train_loss, num_batches, total_students = 0.0, 0, 0
        
        pbar = tqdm(train_loader, desc="Training")
        for batch_idx, (spatial, temporal, targets) in enumerate(pbar):
            spatial = spatial.to(device)
            temporal = temporal.to(device)
            
            try:
                if config.USE_AMP:
                    with autocast():
                        outputs = model(spatial, temporal, targets)
                        loss = criterion(outputs) / config.GRAD_ACCUM_STEPS
                    scaler.scale(loss).backward()
                else:
                    outputs = model(spatial, temporal, targets)
                    loss = criterion(outputs) / config.GRAD_ACCUM_STEPS
                    loss.backward()
            except RuntimeError as e:
                if "out of memory" in str(e):
                    torch.cuda.empty_cache()
                    optimizer.zero_grad()
                    print(f"  ⚠ OOM at batch {batch_idx}")
                    continue
                raise e
            
            total_students += sum(outputs['student_counts'])
            
            if (batch_idx + 1) % config.GRAD_ACCUM_STEPS == 0:
                if config.USE_AMP:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(tst_module.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(tst_module.parameters(), 1.0)
                    optimizer.step()
                
                optimizer.zero_grad()
                scheduler.step()
                global_step += 1
                
                if ema is not None:
                    ema.update()
                
                train_loss += loss.item() * config.GRAD_ACCUM_STEPS
                num_batches += 1
            
            pbar.set_postfix({
                'loss': f'{loss.item() * config.GRAD_ACCUM_STEPS:.4f}',
                'lr': f'{scheduler.get_last_lr()[0]:.2e}',
                'stud': total_students
            })
        
        avg_loss = train_loss / max(num_batches, 1)
        print(f"  Train Loss: {avg_loss:.4f} | Students: {total_students}")
        
        # ══════════════════════════════════════════════════════════════════
        # VALIDATION
        # ══════════════════════════════════════════════════════════════════
        model.tst.eval()
        
        if ema is not None:
            ema.apply_shadow()
        
        all_logits, all_labels = [], []
        
        with torch.no_grad():
            for spatial, temporal, targets in tqdm(val_loader, desc="Validation"):
                spatial = spatial.to(device)
                temporal = temporal.to(device)
                
                if config.USE_AMP:
                    with autocast():
                        outputs = model(spatial, temporal, targets)
                else:
                    outputs = model(spatial, temporal, targets)
                
                if outputs['logits'] is not None:
                    all_logits.append(outputs['logits'].cpu())
                    all_labels.append(outputs['labels'].cpu())
        
        if ema is not None:
            ema.restore()
        
        if all_logits:
            all_logits_cat = torch.cat(all_logits)
            all_labels_cat = torch.cat(all_labels)
            metrics = compute_metrics(all_logits_cat, all_labels_cat)
            
            print(f"  Val mAP: {metrics['mAP']:.4f} | F1: {metrics['f1']:.4f}")
            
            # Per-class AP
            for c in range(15):
                rare = "★" if c in RARE_CLASSES else " "
                print(f"  {rare} {ACTION_NAMES[c]:20s} AP={metrics['aps'][c]:.4f}")
            
            # ══════════════════════════════════════════════════════════════
            # SAVE CHECKPOINT — EVERY EPOCH
            # ══════════════════════════════════════════════════════════════
            tst_state = tst_module.state_dict()
            
            checkpoint_data = {
                'epoch': epoch,
                'global_step': global_step,
                'tst_state_dict': tst_state,
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'scaler_state_dict': scaler.state_dict() if scaler else None,
                'ema_shadow': ema.shadow if ema else None,
                'best_mAP': best_mAP,
                'current_mAP': metrics['mAP'],
                'metrics': metrics,
                'config': {
                    'MODEL_SIZE': config.MODEL_SIZE,
                    'NUM_SEGMENTS': config.NUM_SEGMENTS,
                    'FRAMES_PER_SEGMENT': config.FRAMES_PER_SEGMENT,
                    'TST_WINDOW_SIZE': config.TST_WINDOW_SIZE,
                    'BACKBONE_LR': config.BACKBONE_LR,
                    'HEAD_LR': config.HEAD_LR,
                }
            }
            
            # Save BEST checkpoint
            if metrics['mAP'] > best_mAP:
                best_mAP = metrics['mAP']
                checkpoint_data['best_mAP'] = best_mAP
                torch.save(checkpoint_data, os.path.join(config.OUTPUT_DIR, 'tst_best.pth'))
                print(f"  ✓ NEW BEST! mAP: {best_mAP:.4f}")
            
            # Save LATEST checkpoint (for resume) — EVERY EPOCH
            torch.save(checkpoint_data, os.path.join(config.OUTPUT_DIR, 'tst_latest.pth'))
            print(f"  ✓ Checkpoint saved (epoch {epoch})")
            
            # Also save numbered checkpoint every 5 epochs (for backup)
            if epoch % 5 == 0:
                torch.save(checkpoint_data, os.path.join(config.OUTPUT_DIR, f'tst_epoch_{epoch}.pth'))
    
    print(f"\n{'='*70}")
    print(f"Training complete! Best mAP: {best_mAP:.4f}")
    print("=" * 70)
    
    return best_mAP

# ============================================================================
# MAIN
# ============================================================================
if __name__ == "__main__":
    config = Config()
    # train(config, resume_path="/kaggle/input/models/YOUR_KAGGLE_USERNAME/tst-small-20/pytorch/default/1/tst_best.pth")

# **INFERENCE**

In [5]:
"""
================================================================================
PACR + TST INFERENCE  —  FULL PIPELINE  (v3)
================================================================================
What's new vs original:
  ✓ IoU-based StudentTracker  →  stable S1/S2/S3 IDs across re-detections
  ✓ Polished PIL overlays     →  rounded boxes, pill badges, per-student colours
  ✓ Student photo in card     →  actual crop from last frame
  ✓ EngagementCardGenerator   →  per-student card + class summary dashboard
  ✓ All engagement card code  →  inlined, no separate module needed

Drop this file into /kaggle/working/ and run the last cell.
TST + PACR must already be defined in earlier notebook cells.
================================================================================
"""

# ── standard imports ────────────────────────────────────────────────────────
import os
import sys
import gc
import math
import shutil
import warnings
import random
from io import BytesIO
from collections import defaultdict

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw, ImageFont
from torchvision import transforms
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')

# ── PACR import ──────────────────────────────────────────────────────────────
pacr_source = "/kaggle/input/datasets/YOUR_KAGGLE_USERNAME/pacr-detr/pacr-detr.py"
shutil.copy(pacr_source, "/kaggle/working/pacr_detr.py")
sys.path.insert(0, '/kaggle/working')
from pacr_detr import PACRDETRDetector, PACRDETRInference, Config as PACRConfig
print("✓ PACR imported!")

# ── TST, Config, ACTION_NAMES assumed defined in a previous notebook cell ───
# (TST, VideoSwinTransformer, Config, ACTION_NAMES, etc.)


# ============================================================================
# PATHS & HYPERPARAMETERS
# ============================================================================

PACR_CHECKPOINT    = "/kaggle/input/models/YOUR_KAGGLE_USERNAME/pacr-bound/pytorch/default/1/best (1) (1).pth"
TST_CHECKPOINT     = "/kaggle/input/models/YOUR_KAGGLE_USERNAME/tst-small-best/pytorch/default/1/tst_best (1).pth"
INPUT_VIDEO        = "/kaggle/input/datasets/YOUR_KAGGLE_USERNAME/sav-dataset/SAV/data/sav/video_clips/20181016_05_1064.mp4"
OUTPUT_VIDEO       = "/kaggle/working/output_annotated.mp4"
CARDS_DIR          = "/kaggle/working/engagement_cards"

DEVICE             = "cuda" if torch.cuda.is_available() else "cpu"

# Detection
DETECTION_INTERVAL = 30      # re-run PACR every N frames
CONF_THRESHOLD     = 0.45
NMS_THRESHOLD      = 0.50

# Classification
NUM_SEGMENTS       = 3
FRAMES_PER_SEG     = 16
BUFFER_LEN         = NUM_SEGMENTS * FRAMES_PER_SEG   # 48
SPATIAL_SIZE       = 224
CLASSIFY_INTERVAL  = 16
MAX_BATCH_TST      = 4

# Visualisation
ACTION_THRESHOLD   = 0.40
MAX_LABELS_SHOWN   = 3


# ============================================================================
# ACTION METADATA
# ============================================================================

ACTION_NAMES_DISPLAY = [
    'Sitting',        'Standing',       'Looking forward', 'Looking sideways', 'Reading',
    'Flipping books', 'Touching sth',   'Raising hand',    'Hands down',       'Taking notes',
    'Applauding',     'Bending',        'Turning around',  'Talking',          'Answering',
]
# Short names for original inference loop compatibility
ACTION_NAMES_ORIG = [
    'sit', 'stand', 'look_forward', 'look_sideways', 'read',
    'flip_books', 'touch_sth', 'raise_hand', 'hands_down', 'take_notes',
    'applaud', 'bend', 'turn_around', 'talk_with_others', 'answer_questions'
]
ACTION_SHORT = [
    'Sit','Stand','Fwd','Side','Read',
    'Flip','Touch','Hand+','Hand-','Notes',
    'Clap','Bend','Turn','Talk','Ans',
]
GROUPS = {
    'Posture':   [0, 1, 11, 12],
    'Attention': [2, 3, 7, 8],
    'Activity':  [4, 5, 6, 9],
    'Social':    [10, 13, 14],
}
GROUP_COLORS = {
    'Posture':   '#5C72F0',
    'Attention': '#14A87A',
    'Activity':  '#F5A623',
    'Social':    '#E05A7B',
}
# Matches the slide formula exactly
# Order: Sit, Stand, LookFwd, LookSide, Read, Flip, Touch, RaiseHand, HandsDown, Notes, Clap, Bend, TurnAround, Talk, Answer
ENGAGE_WEIGHTS = np.array([
     0.3,   # sit
     0.0,   # stand
     0.8,   # look_forward
    -0.6,   # look_sideways
     0.9,   # read
     0.7,   # flip_books
     0.0,   # touch_sth
     1.0,   # raise_hand
     0.0,   # hands_down
     0.9,   # take_notes
     0.0,   # applaud
    -0.2,   # bend
    -0.7,   # turn_around
    -0.8,   # talk_with_others
     1.0,   # answer_questions
])

M = np.sum(np.abs(ENGAGE_WEIGHTS))   # = 7.6

# Replace the es= line in generate_student_card and generate_class_summary:


# Per-student box palette  (stroke_hex, fill_hex)
BOX_PALETTE = [
    ('#5C72F0', '#EEF0FE'),
    ('#14A87A', '#E6F9F3'),
    ('#F5A623', '#FEF7E6'),
    ('#E05A7B', '#FDE8EF'),
    ('#8B5CF6', '#F0EBFD'),
    ('#0EA5E9', '#E0F5FE'),
    ('#10B981', '#D1FAE5'),
    ('#F97316', '#FFF0E6'),
]

# ── card palette ─────────────────────────────────────────────
BG       = '#F7F8FC'
CARD_BG  = '#FFFFFF'
BORDER   = '#E4E7EF'
TEXT_PRI = '#1A1D2E'
TEXT_SEC = '#6B7280'
ACCENT   = '#5C72F0'
GREEN    = '#14A87A'
AMBER    = '#F5A623'
RED      = '#E05A7B'


# ============================================================================
# FONT HELPER
# ============================================================================

def _font(size, bold=False):
    candidates = [
        '/usr/share/fonts/truetype/dejavu/DejaVuSans{}.ttf',
        '/usr/share/fonts/truetype/liberation/LiberationSans{}-Regular.ttf',
        '/usr/share/fonts/truetype/freefont/FreeSans{}.ttf',
    ]
    suffix = '-Bold' if bold else ''
    for tmpl in candidates:
        path = tmpl.format(suffix)
        if os.path.exists(path):
            try:
                return ImageFont.truetype(path, size)
            except Exception:
                pass
    return ImageFont.load_default()

def _rrect(draw, xy, radius, fill=None, outline=None, width=1):
    draw.rounded_rectangle(list(xy), radius=radius,
                            fill=fill, outline=outline, width=width)

def _engagement_color(s):
    return GREEN if s >= 0.65 else (AMBER if s >= 0.45 else RED)

def _engagement_label(s):
    return 'Engaged' if s >= 0.65 else ('Moderate' if s >= 0.45 else 'Disengaged')


# ============================================================================
# IoU-BASED STUDENT TRACKER
# ============================================================================

class StudentTracker:
    """
    Keeps stable integer IDs across PACR re-detections via IoU matching.

    Usage
    -----
        tracker = StudentTracker()
        # every time PACR fires:
        current_boxes, track_ids = tracker.update(boxes_xyxy)
        # track_ids[i] is a stable 1-indexed int for student i
    """

    def __init__(self, iou_threshold=0.40, max_missing=5):
        self.iou_thr     = iou_threshold
        self.max_missing = max_missing
        self._tracks     = {}   # id → {'box':(4,), 'missing':int}
        self._next_id    = 1

    def update(self, boxes):
        """
        boxes : (N,4) float32 normalised xyxy  OR  None/empty array
        Returns (boxes, ids) where ids is (N,) stable int array.
        """
        if boxes is None or len(boxes) == 0:
            self._age_tracks()
            return np.zeros((0, 4), dtype=np.float32), np.zeros(0, dtype=int)

        boxes = np.asarray(boxes, dtype=np.float32)
        N = len(boxes)

        if not self._tracks:
            return boxes, self._register_all(boxes)

        track_ids   = list(self._tracks.keys())
        track_boxes = np.stack([self._tracks[t]['box'] for t in track_ids])
        iou_mat     = self._iou_matrix(boxes, track_boxes)   # (N, M)

        assigned_det, assigned_trk = {}, set()
        for i, j, iou in sorted(
                [(i, j, iou_mat[i, j]) for i in range(N) for j in range(len(track_ids))],
                key=lambda x: -x[2]):
            if iou < self.iou_thr:
                break
            if i in assigned_det or j in assigned_trk:
                continue
            tid = track_ids[j]
            assigned_det[i] = tid
            assigned_trk.add(j)
            self._tracks[tid]['box']     = boxes[i]
            self._tracks[tid]['missing'] = 0

        ids = np.zeros(N, dtype=int)
        for i in range(N):
            if i in assigned_det:
                ids[i] = assigned_det[i]
            else:
                new_id = self._next_id; self._next_id += 1
                self._tracks[new_id] = {'box': boxes[i], 'missing': 0}
                ids[i] = new_id

        # age/prune unmatched tracks
        for j, t in enumerate(track_ids):
            if j not in assigned_trk:
                self._tracks[t]['missing'] += 1
                if self._tracks[t]['missing'] > self.max_missing:
                    del self._tracks[t]

        return boxes, ids

    # ── internal ────────────────────────────────────────────
    def _register_all(self, boxes):
        ids = []
        for box in boxes:
            tid = self._next_id; self._next_id += 1
            self._tracks[tid] = {'box': box, 'missing': 0}
            ids.append(tid)
        return np.array(ids, dtype=int)

    def _age_tracks(self):
        for t in list(self._tracks):
            self._tracks[t]['missing'] += 1
            if self._tracks[t]['missing'] > self.max_missing:
                del self._tracks[t]

    @staticmethod
    def _iou_matrix(a, b):
        ax1,ay1,ax2,ay2 = a[:,0],a[:,1],a[:,2],a[:,3]
        bx1,by1,bx2,by2 = b[:,0],b[:,1],b[:,2],b[:,3]
        ix1 = np.maximum(ax1[:,None], bx1[None,:])
        iy1 = np.maximum(ay1[:,None], by1[None,:])
        ix2 = np.minimum(ax2[:,None], bx2[None,:])
        iy2 = np.minimum(ay2[:,None], by2[None,:])
        inter  = np.maximum(0, ix2-ix1) * np.maximum(0, iy2-iy1)
        area_a = (ax2-ax1)*(ay2-ay1)
        area_b = (bx2-bx1)*(by2-by1)
        return inter / (area_a[:,None] + area_b[None,:] - inter + 1e-6)


# ============================================================================
# FRAME / TUBE UTILITIES
# ============================================================================

_normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std =[0.229, 0.224, 0.225]
)

def frame_to_tensor(bgr_frame: np.ndarray) -> torch.Tensor:
    """BGR ndarray → normalised (3,H,W) float32 tensor."""
    rgb = cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB)
    t   = torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0
    return _normalize(t)


def extract_tube(frame_buffer: list,
                 box_norm: np.ndarray,
                 spatial_size: int = 224) -> torch.Tensor:
    """
    frame_buffer : list of (3,H,W) normalised tensors
    box_norm     : [x1,y1,x2,y2] in [0,1]
    returns      : (K*T, 3, spatial_size, spatial_size)
    """
    crops = []
    for f in frame_buffer:
        _, H, W = f.shape
        x1 = max(0,   int(box_norm[0] * W))
        y1 = max(0,   int(box_norm[1] * H))
        x2 = min(W-1, int(box_norm[2] * W))
        y2 = min(H-1, int(box_norm[3] * H))
        if x2 <= x1 or y2 <= y1:
            crop = torch.zeros(3, spatial_size, spatial_size)
        else:
            crop = f[:, y1:y2, x1:x2]
            crop = F.interpolate(crop.unsqueeze(0),
                                 size=(spatial_size, spatial_size),
                                 mode='bilinear', align_corners=False).squeeze(0)
        crops.append(crop)
    return torch.stack(crops, dim=0)


def crop_student_pil(bgr_frame: np.ndarray,
                     box_norm: np.ndarray,
                     padding: float = 0.04) -> Image.Image:
    """Crop a student from a BGR frame and return a PIL image."""
    H, W = bgr_frame.shape[:2]
    x1 = max(0, int((box_norm[0] - padding) * W))
    y1 = max(0, int((box_norm[1] - padding) * H))
    x2 = min(W, int((box_norm[2] + padding) * W))
    y2 = min(H, int((box_norm[3] + padding) * H))
    crop = bgr_frame[y1:y2, x1:x2]
    if crop.size == 0:
        return Image.new('RGB', (120, 180), '#CCCCCC')
    return Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))


# ============================================================================
# MODEL LOADERS
# ============================================================================

def load_pacr(checkpoint_path: str, device: str):
    pacr_config = PACRConfig()
    inferencer  = PACRDETRInference(checkpoint_path, pacr_config, device='cpu')
    inferencer.model    = inferencer.model.to(device)
    inferencer.device   = torch.device(device)
    print(f"✓ PACR loaded  →  device={device}")
    return inferencer, pacr_config


def load_tst(checkpoint_path: str, device: str):
    tst_config = Config()        # defined in earlier notebook cell
    model      = TST(tst_config)
    ckpt  = torch.load(checkpoint_path, map_location=device, weights_only=False)
    state = ckpt.get('tst_state_dict', ckpt)
    state = {k.replace('module.', ''): v for k, v in state.items()}
    model.load_state_dict(state, strict=False)
    model.to(device).eval()
    print(f"✓ TST loaded   →  epoch={ckpt.get('epoch','?')}  "
          f"best_mAP={ckpt.get('best_mAP','?')}")
    return model


# ============================================================================
# TST CLASSIFICATION HELPER
# ============================================================================

@torch.no_grad()
def classify_students(tst_model,
                      frame_buffer: list,
                      boxes_norm: np.ndarray,
                      device: str) -> list:
    """
    Returns list of dicts:
        {'box': ..., 'labels': [...], 'scores': [...], 'probs': np.ndarray(15,)}
    """
    if len(boxes_norm) == 0:
        return []

    # build tubes on CPU
    tubes = [extract_tube(frame_buffer, box, SPATIAL_SIZE) for box in boxes_norm]

    all_probs = []
    for i in range(0, len(tubes), MAX_BATCH_TST):
        batch  = torch.stack(tubes[i:i+MAX_BATCH_TST], dim=0).to(device)
        logits = tst_model(batch)
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        del batch, logits
        torch.cuda.empty_cache()

    all_probs = np.concatenate(all_probs, axis=0)   # (N, 15)

    results = []
    for box, prob in zip(boxes_norm, all_probs):
        top_idx = np.argsort(prob)[::-1]
        labels, scores = [], []
        for idx in top_idx[:MAX_LABELS_SHOWN]:
            if prob[idx] >= ACTION_THRESHOLD:
                labels.append(ACTION_NAMES_DISPLAY[idx])
                scores.append(float(prob[idx]))
        results.append({'box': box, 'labels': labels,
                        'scores': scores, 'probs': prob})
    return results


# ============================================================================
# VIDEO FRAME ANNOTATION  (PIL-based, polished)
# ============================================================================

def annotate_video_frame(bgr_frame: np.ndarray,
                          student_results: list,
                          frame_idx: int,
                          track_ids=None) -> np.ndarray:
    """
    Draw rounded boxes, stable ID badges, and action pill badges on the frame.

    Returns annotated BGR ndarray (same size as input, ready for VideoWriter).
    """
    rgb = cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(rgb)
    draw = ImageDraw.Draw(img, 'RGBA')
    IW, IH = img.size

    f_id  = _font(13, bold=True)
    f_act = _font(11)

    for i, res in enumerate(student_results):
        box    = res.get('box',    np.zeros(4))
        labels = res.get('labels', [])
        scores = res.get('scores', [])
        tid    = int(track_ids[i]) if (track_ids is not None and i < len(track_ids)) else i+1
        stroke, pill_bg = BOX_PALETTE[(tid - 1) % len(BOX_PALETTE)]

        x1 = int(box[0]*IW); y1 = int(box[1]*IH)
        x2 = int(box[2]*IW); y2 = int(box[3]*IH)

        # semi-transparent fill + border
        draw.rounded_rectangle([x1, y1, x2, y2], radius=6,
                                fill=stroke + '22', outline=stroke, width=2)

        # ID badge  (top-left corner, sits just above the box)
        badge = f'S{tid}'
        bb    = draw.textbbox((0, 0), badge, font=f_id)
        bw    = bb[2]-bb[0]+14
        bh    = bb[3]-bb[1]+8
        _rrect(draw, [x1, max(0, y1-bh-2), x1+bw, y1], radius=5, fill=stroke)
        draw.text((x1+7, max(bh//2, y1-bh//2-2)), badge,
                  font=f_id, fill='#FFFFFF', anchor='lm')

        # action pills  (stacked below the box)
        py = y2 + 4
        for lbl, sc in zip(labels[:3], scores[:3]):
            txt = f'{lbl}  {int(sc*100)}%'
            bb  = draw.textbbox((0, 0), txt, font=f_act)
            pw  = bb[2]-bb[0]+16
            ph  = bb[3]-bb[1]+8
            _rrect(draw, [x1, py, x1+pw, py+ph],
                   radius=5, fill=pill_bg, outline=stroke, width=1)
            draw.text((x1+8, py+ph//2), txt,
                      font=f_act, fill=stroke, anchor='lm')
            py += ph + 4

    # HUD strip
    draw.rectangle([0, 0, IW, 36], fill='#000000BB')
    draw.text((12, 18), f'Frame {frame_idx:05d}',
              font=_font(12), fill='#FFFFFF', anchor='lm')
    draw.text((IW-12, 18), f'{len(student_results)} students',
              font=_font(12), fill='#CCCCCC', anchor='rm')

    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)


# ============================================================================
# STUDENT ENGAGEMENT CARD
# ============================================================================

def generate_student_card(student_id: int,
                           probs: np.ndarray,
                           student_img: Image.Image = None,
                           out_path: str = None,
                           stroke_color: str = None) -> Image.Image:
    """
    Renders a 700×400 PNG report card for one student.

    Left panel  : photo (or coloured avatar) + engagement ring gauge
    Right panel : group bars, top-action tags, all-15 mini-bars
    """
    W, H = 700, 400
    img  = Image.new('RGB', (W, H), BG)
    draw = ImageDraw.Draw(img)
    accent = stroke_color or ACCENT

    # outer card
    _rrect(draw, [14, 14, W-14, H-14], radius=18,
           fill=CARD_BG, outline=BORDER, width=1)

    # ── LEFT PANEL ──────────────────────────────────────────────────────────
    PW = 178
    _rrect(draw, [26, 26, 26+PW, H-26], radius=14, fill=BG)

    PHOTO_W, PHOTO_H = 130, 160
    px = 26 + (PW - PHOTO_W) // 2
    py = 38

    if student_img is not None:
        thumb = student_img.copy().convert('RGB')
        ratio = max(PHOTO_W / max(thumb.width, 1),
                    PHOTO_H / max(thumb.height, 1))
        nw = max(1, int(thumb.width  * ratio))
        nh = max(1, int(thumb.height * ratio))
        thumb = thumb.resize((nw, nh), Image.LANCZOS)
        left  = (thumb.width  - PHOTO_W) // 2
        top   = (thumb.height - PHOTO_H) // 2
        thumb = thumb.crop((left, top, left+PHOTO_W, top+PHOTO_H))

        # rounded-rect mask
        mask = Image.new('L', (PHOTO_W, PHOTO_H), 0)
        ImageDraw.Draw(mask).rounded_rectangle(
            [0, 0, PHOTO_W, PHOTO_H], radius=10, fill=255)
        thumb_rgba = thumb.convert('RGBA')
        thumb_rgba.putalpha(mask)
        img.paste(thumb_rgba, (px, py), thumb_rgba)
        _rrect(draw, [px-2, py-2, px+PHOTO_W+2, py+PHOTO_H+2],
               radius=11, outline=accent, width=2)
    else:
        # coloured avatar fallback
        _rrect(draw, [px, py, px+PHOTO_W, py+PHOTO_H], radius=10, fill=accent)
        f_av = _font(44, bold=True)
        lbl  = f'S{student_id}'
        bb   = draw.textbbox((0, 0), lbl, font=f_av)
        tw, th = bb[2]-bb[0], bb[3]-bb[1]
        draw.text((px+(PHOTO_W-tw)//2, py+(PHOTO_H-th)//2-2),
                  lbl, font=f_av, fill='#FFFFFF')

    # name label
    draw.text((26+PW//2, py+PHOTO_H+14), f'Student {student_id}',
              font=_font(13, bold=True), fill=TEXT_PRI, anchor='mm')

    # engagement ring
    es = float((np.dot(probs, ENGAGE_WEIGHTS) + M) / (2 * M))
    e_color = _engagement_color(es)
    e_label = _engagement_label(es)
    rcx     = 26 + PW//2
    rcy     = py + PHOTO_H + 54
    rr      = 34
    draw.arc([rcx-rr, rcy-rr, rcx+rr, rcy+rr],
             start=0, end=360, fill=BORDER, width=7)
    sweep = max(1, int(es * 360))
    draw.arc([rcx-rr, rcy-rr, rcx+rr, rcy+rr],
             start=-90, end=-90+sweep, fill=e_color, width=7)
    pct_txt = f'{int(es*100)}%'
    bb = draw.textbbox((0,0), pct_txt, font=_font(15, bold=True))
    draw.text((rcx-(bb[2]-bb[0])//2, rcy-11), pct_txt,
              font=_font(15, bold=True), fill=e_color)
    draw.text((rcx, rcy+12), e_label,
              font=_font(9), fill=e_color, anchor='mm')

    # ── RIGHT PANEL ──────────────────────────────────────────────────────────
    RX = 26 + PW + 18
    RW = W - RX - 26
    f_h = _font(11, bold=True)
    f_s = _font(10)
    f_t = _font(11)

    # group bars
    draw.text((RX, 34), 'Behaviour breakdown', font=f_h, fill=TEXT_PRI)
    gy = 54
    for gname, gidxs in GROUPS.items():
        gscore   = float(np.mean(probs[gidxs]))
        gc       = GROUP_COLORS[gname]
        bar_full = RW - 92
        bar_fill = max(0, int(gscore * bar_full))
        draw.text((RX, gy), gname, font=f_s, fill=TEXT_SEC)
        tx = RX + 82
        _rrect(draw, [tx, gy+2, tx+bar_full, gy+11], radius=4, fill=BORDER)
        if bar_fill > 0:
            _rrect(draw, [tx, gy+2, tx+bar_fill, gy+11], radius=4, fill=gc)
        draw.text((tx+bar_full+5, gy+2), f'{int(gscore*100)}%',
                  font=_font(9), fill=gc)
        gy += 22

    draw.line([(RX, gy+4), (W-30, gy+4)], fill=BORDER, width=1)
    gy += 14

    # top action tags
    draw.text((RX, gy), 'Top detected actions', font=f_h, fill=TEXT_PRI)
    gy += 18
    top_idxs = np.argsort(probs)[::-1][:8]
    TAG_H, TAG_PAD = 22, 9
    tx, ty = RX, gy
    for idx in top_idxs:
        p = probs[idx]
        if p < 0.22:
            break
        name = ACTION_NAMES_DISPLAY[idx]
        bb   = draw.textbbox((0, 0), name, font=f_t)
        tw_t = bb[2]-bb[0]+TAG_PAD*2
        if tx + tw_t > W - 30:
            tx = RX; ty += TAG_H + 5
        if p >= 0.65:
            bg_c, txt_c = '#E6F9F3', '#0D7A57'
        elif p >= 0.40:
            bg_c, txt_c = '#FEF3E2', '#B87100'
        else:
            bg_c, txt_c = '#F1F3FE', '#3A50C4'
        _rrect(draw, [tx, ty, tx+tw_t, ty+TAG_H], radius=5, fill=bg_c)
        draw.text((tx+TAG_PAD, ty+TAG_H//2), name,
                  font=f_t, fill=txt_c, anchor='lm')
        tx += tw_t + 6
    gy = ty + TAG_H + 14

    draw.line([(RX, gy), (W-30, gy)], fill=BORDER, width=1)
    gy += 12

    # all-15 mini-bars  (3 columns)
    draw.text((RX, gy), 'All actions', font=f_h, fill=TEXT_PRI)
    gy += 14
    col_w = RW // 3
    for i, (sname, p) in enumerate(zip(ACTION_SHORT, probs)):
        cx = RX + (i % 3) * col_w
        cy = gy + (i // 3) * 17
        draw.text((cx, cy), sname, font=_font(9), fill=TEXT_SEC)
        bx    = cx + 44
        bfull = col_w - 52
        _rrect(draw, [bx, cy+2, bx+bfull, cy+9], radius=3, fill=BORDER)
        fw = max(2, int(p * bfull))
        fc = GREEN if p >= 0.5 else (AMBER if p >= 0.3 else BORDER)
        _rrect(draw, [bx, cy+2, bx+fw, cy+9], radius=3, fill=fc)

    draw.text((W-22, H-12), 'SAV · TST', font=_font(8), fill=BORDER, anchor='rs')

    if out_path:
        img.save(out_path, quality=95)
    return img


# ============================================================================
# CLASS SUMMARY DASHBOARD  (matplotlib)
# ============================================================================

def generate_class_summary(all_probs: np.ndarray,
                            out_path: str = None,
                            video_id: str = '') -> Image.Image:
    """
    4-panel dashboard:
      top-left  : group radar
      top-right : per-action horizontal bar (sorted)
      bot-left  : student × action heatmap
      bot-right : engagement donut
    """
    N   = all_probs.shape[0]
    fig = plt.figure(figsize=(13, 8.5), facecolor=BG, dpi=120)
    fig.patch.set_facecolor(BG)

    gs = fig.add_gridspec(2, 3, hspace=0.44, wspace=0.32,
                          left=0.06, right=0.97, top=0.88, bottom=0.07)
    ax_radar  = fig.add_subplot(gs[0, 0], polar=True)
    ax_bar    = fig.add_subplot(gs[0, 1:])
    ax_heat   = fig.add_subplot(gs[1, :2])
    ax_engage = fig.add_subplot(gs[1, 2])

    fig.text(0.5, 0.95, 'Classroom Engagement Report',
             ha='center', fontsize=16, fontweight='bold', color=TEXT_PRI)
    sub = f'{N} students detected'
    if video_id:
        sub = f'Clip: {video_id}  -  ' + sub
    fig.text(0.5, 0.915, sub, ha='center', fontsize=10, color=TEXT_SEC)

    # ── radar ────────────────────────────────────────────────
    gs_scores = {g: float(np.mean(all_probs[:, idx]))
                 for g, idx in GROUPS.items()}
    labels  = list(gs_scores.keys())
    values  = list(gs_scores.values()) + [list(gs_scores.values())[0]]
    angles  = [n / len(labels) * 2 * math.pi for n in range(len(labels))] + [0]
    ax_radar.set_facecolor(CARD_BG)
    ax_radar.plot(angles, values, 'o-', lw=2, color=ACCENT)
    ax_radar.fill(angles, values, alpha=0.18, color=ACCENT)
    ax_radar.set_xticks(angles[:-1])
    ax_radar.set_xticklabels(labels, fontsize=9, color=TEXT_PRI)
    ax_radar.tick_params(axis='x', pad=18)   # ← ADD THIS LINE
    ax_radar.set_yticks([0.25, 0.5, 0.75])
    ax_radar.set_yticklabels(['25%','50%','75%'], fontsize=7, color=TEXT_SEC)
    ax_radar.set_ylim(0, 1)
    ax_radar.spines['polar'].set_color(BORDER)
    ax_radar.set_title('Group breakdown', fontsize=11, color=TEXT_PRI, pad=12)
    ax_radar.grid(color=BORDER, lw=0.8)

    # ── horizontal bar ────────────────────────────────────────
    mean_p  = all_probs.mean(axis=0)
    idx_s   = np.argsort(mean_p)[::-1]
    names_s = [ACTION_NAMES_DISPLAY[i] for i in idx_s]
    vals_s  = [mean_p[i] for i in idx_s]
    clrs_b  = [GREEN if v>=0.6 else (AMBER if v>=0.3 else '#D0D5E8')
               for v in vals_s]
    ax_bar.set_facecolor(CARD_BG)
    bars = ax_bar.barh(range(len(names_s)), vals_s,
                       color=clrs_b, height=0.65, zorder=2)
    ax_bar.set_yticks(range(len(names_s)))
    ax_bar.set_yticklabels(names_s, fontsize=9, color=TEXT_PRI)
    ax_bar.set_xlim(0, 1.08)
    ax_bar.set_xlabel('Mean probability', fontsize=9, color=TEXT_SEC)
    ax_bar.set_title('Class action frequencies', fontsize=11, color=TEXT_PRI)
    ax_bar.axvline(0.5, color=BORDER, ls='--', lw=0.8, zorder=1)
    for bar, val in zip(bars, vals_s):
        ax_bar.text(val+0.01, bar.get_y()+bar.get_height()/2,
                    f'{val:.0%}', va='center', fontsize=8, color=TEXT_SEC)
    ax_bar.tick_params(labelsize=8, color=BORDER)
    ax_bar.spines[['top','right']].set_visible(False)
    ax_bar.spines[['left','bottom']].set_color(BORDER)
    ax_bar.invert_yaxis()

    # ── heatmap ──────────────────────────────────────────────
    ax_heat.set_facecolor(CARD_BG)
    im = ax_heat.imshow(all_probs, aspect='auto', cmap='YlGn',
                        vmin=0, vmax=1, interpolation='nearest')
    ax_heat.set_xticks(range(15))
    ax_heat.set_xticklabels(ACTION_SHORT, fontsize=8, color=TEXT_PRI,
                             rotation=35, ha='right')
    ax_heat.set_yticks(range(N))
    ax_heat.set_yticklabels([f'S{i+1}' for i in range(N)],
                             fontsize=8, color=TEXT_PRI)
    ax_heat.set_title('Student x action heatmap', fontsize=11, color=TEXT_PRI)
    ax_heat.tick_params(length=0)
    ax_heat.set_xticks(np.arange(-0.5, 15, 1), minor=True)
    ax_heat.set_yticks(np.arange(-0.5, N,  1), minor=True)
    ax_heat.grid(which='minor', color='white', lw=0.5)
    cbar = fig.colorbar(im, ax=ax_heat, shrink=0.85, pad=0.02)
    cbar.ax.tick_params(labelsize=7)
    cbar.set_label('Probability', fontsize=8, color=TEXT_SEC)

    # ── donut ────────────────────────────────────────────────
    scores     = (all_probs * ENGAGE_WEIGHTS).sum(1) / ENGAGE_WEIGHTS.sum()
    engaged    = int((scores >= 0.65).sum())
    moderate   = int(((scores >= 0.35) & (scores < 0.65)).sum())
    disengaged = int((scores < 0.35).sum())
    ax_engage.set_facecolor(CARD_BG)
    _, _, autotexts = ax_engage.pie(
        [engaged, moderate, disengaged],
        colors=[GREEN, AMBER, RED], autopct='%1.0f%%',
        startangle=90, pctdistance=0.78,
        wedgeprops=dict(width=0.48, edgecolor='white', linewidth=2)
    )
    for at in autotexts:
        at.set_fontsize(9); at.set_color('white'); at.set_fontweight('bold')
    ax_engage.set_title('Engagement', fontsize=11, color=TEXT_PRI)
    patches = [mpatches.Patch(color=c, label=f'{l} ({s})')
               for c,l,s in zip([GREEN,AMBER,RED],
                                  ['Engaged','Moderate','Disengaged'],
                                  [engaged, moderate, disengaged])]
    ax_engage.legend(handles=patches, fontsize=8, loc='lower center',
                     bbox_to_anchor=(0.5, -0.22), frameon=False,
                     labelcolor=TEXT_PRI)
    ax_engage.text(0, 0, f'{N}\nstudents', ha='center', va='center',
                   fontsize=9, color=TEXT_PRI, linespacing=1.4)

    buf = BytesIO()
    fig.savefig(buf, dpi=120, bbox_inches='tight', facecolor=BG)
    plt.close(fig)
    buf.seek(0)
    pil = Image.open(buf).convert('RGB')
    if out_path:
        pil.save(out_path, quality=95)
    return pil


# ============================================================================
# ENGAGEMENT CARD GENERATOR
# ============================================================================

class EngagementCardGenerator:
    def __init__(self, output_dir: str):
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        print(f'✓ EngagementCardGenerator → {output_dir}')

    def generate_all_cards(self, all_probs: np.ndarray,
                            student_images: list = None,
                            track_ids=None):
        N = all_probs.shape[0]
        for i in range(N):
            tid     = int(track_ids[i]) if track_ids is not None else i+1
            img_pil = student_images[i] if (student_images and i < len(student_images)) else None
            stroke, _ = BOX_PALETTE[(tid-1) % len(BOX_PALETTE)]
            path = os.path.join(self.output_dir, f'student_{tid:02d}_card.png')
            generate_student_card(tid, all_probs[i], img_pil, path, stroke)
        print(f'  ✓ {N} student cards saved → {self.output_dir}')

    def generate_class_summary(self, all_probs: np.ndarray,
                                video_id: str = '') -> str:
        path = os.path.join(self.output_dir, 'class_summary.png')
        generate_class_summary(all_probs, path, video_id)
        print(f'  ✓ Class summary saved → {path}')
        return path


# ============================================================================
# MAIN INFERENCE LOOP
# ============================================================================

def run_inference(input_path: str, output_path: str):

    # ── load models ────────────────────────────────────────────────────────
    pacr_inferencer, _ = load_pacr(PACR_CHECKPOINT, DEVICE)
    tst_model          = load_tst(TST_CHECKPOINT,   DEVICE)

    # ── tracker + generator ────────────────────────────────────────────────
    tracker   = StudentTracker(iou_threshold=0.40, max_missing=5)
    generator = EngagementCardGenerator(CARDS_DIR)

    # ── open video ─────────────────────────────────────────────────────────
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {input_path}")

    fps   = cap.get(cv2.CAP_PROP_FPS) or 25.0
    VW    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    VH    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    vid   = os.path.splitext(os.path.basename(input_path))[0]
    print(f"Video: {VW}x{VH}  {fps:.1f}fps  {total} frames  id={vid}")

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_path, fourcc, fps, (VW, VH))

    # ── state ──────────────────────────────────────────────────────────────
    frame_buffer    = []               # rolling list of (3,H,W) tensors
    current_boxes   = np.zeros((0, 4), dtype=np.float32)
    track_ids       = np.zeros(0, dtype=int)
    student_results = []               # latest TST classify_students output
    last_frame      = None             # last BGR frame (for card crops)
    frame_idx       = 0

    print(f"\nStarting inference "
          f"(detect every {DETECTION_INTERVAL}f, "
          f"classify every {CLASSIFY_INTERVAL}f) ...\n")

    while True:
        ret, bgr = cap.read()
        if not ret:
            break
        last_frame = bgr.copy()
        frame_idx += 1
        pil_frame  = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))

        # ── PACR detection ────────────────────────────────────────────────
        if frame_idx == 1 or (frame_idx % DETECTION_INTERVAL == 0):
            boxes_raw, _ = pacr_inferencer.detect_with_nms(
                pil_frame,
                conf_threshold=CONF_THRESHOLD,
                nms_threshold=NMS_THRESHOLD,
                return_format='xyxy'
            )
            # ← tracker gives back SAME boxes but with STABLE IDs
            current_boxes, track_ids = tracker.update(boxes_raw)
            if frame_idx == 1:
                print(f"  Frame {frame_idx:5d}  detected {len(current_boxes)} students"
                      f"  IDs={track_ids.tolist()}")

        # ── rolling frame buffer ──────────────────────────────────────────
        tensor = frame_to_tensor(bgr)
        frame_buffer.append(tensor)
        if len(frame_buffer) > BUFFER_LEN:
            frame_buffer.pop(0)

        # ── TST classification ────────────────────────────────────────────
        if (len(frame_buffer) >= FRAMES_PER_SEG
                and len(current_boxes) > 0
                and (frame_idx % CLASSIFY_INTERVAL == 0
                     or frame_idx == FRAMES_PER_SEG)):

            buf = frame_buffer
            if len(buf) < BUFFER_LEN:
                pad  = BUFFER_LEN - len(buf)
                buf  = buf[0:1] * pad + buf

            student_results = classify_students(
                tst_model, buf, current_boxes, DEVICE)

            if frame_idx % 100 == 0:
                print(f"  Frame {frame_idx:5d}  {len(student_results)} students classified")
                for i, r in enumerate(student_results[:3]):
                    tid = int(track_ids[i]) if i < len(track_ids) else i+1
                    print(f"    S{tid}: {r['labels']}")

        # ── build display list (box index aligned with track_ids) ─────────
        display_results = []
        for i, box in enumerate(current_boxes):
            if i < len(student_results):
                res        = student_results[i].copy()
                res['box'] = box      # refresh box to latest PACR position
            else:
                res = {'box': box, 'labels': [], 'scores': [], 'probs': np.zeros(15)}
            display_results.append(res)

        # ── annotate frame (polished PIL) ─────────────────────────────────
        annotated = annotate_video_frame(bgr, display_results,
                                          frame_idx, track_ids)
        writer.write(annotated)

    # ── end-of-video: generate engagement cards ───────────────────────────
    if last_frame is not None and len(current_boxes) > 0:
        print("\nGenerating engagement cards ...")

        # crop actual student images from last frame
        student_images = [crop_student_pil(last_frame, box)
                          for box in current_boxes]

        # final classification on the last full buffer
        buf = frame_buffer
        if len(buf) < BUFFER_LEN:
            buf = buf[0:1] * (BUFFER_LEN - len(buf)) + buf

        tubes     = [extract_tube(buf, box, SPATIAL_SIZE) for box in current_boxes]
        all_probs = []
        for i in range(0, len(tubes), MAX_BATCH_TST):
            batch = torch.stack(tubes[i:i+MAX_BATCH_TST], dim=0).to(DEVICE)
            with torch.no_grad():
                logits = tst_model(batch)
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            del batch, logits
            torch.cuda.empty_cache()
        all_probs = np.concatenate(all_probs, axis=0)   # (N, 15)

        generator.generate_all_cards(all_probs,
                                      student_images=student_images,
                                      track_ids=track_ids)
        generator.generate_class_summary(all_probs, video_id=vid)

    cap.release()
    writer.release()
    print(f"\n✓ Done!  Annotated video → {output_path}")
    print(f"  Total frames processed : {frame_idx}")
    print(f"  Engagement cards       → {CARDS_DIR}/")


# ============================================================================
# RUN
# ============================================================================

gc.collect()
torch.cuda.empty_cache()

run_inference(INPUT_VIDEO, OUTPUT_VIDEO)

from IPython.display import Video
Video(OUTPUT_VIDEO, width=800)

# **GUI**

In [8]:
"""
================================================================================
CLASSROOM BEHAVIOR RECOGNITION — POLISHED GRADIO GUI
================================================================================
Integrates PACR-DETR + TST inference pipeline with:
  • Annotated video output with polished PIL overlays
  • Per-student engagement report cards with cropped photos
  • Class-level summary dashboard (radar, heatmap, donut, bars)
  • Single frame detection mode
  • Sample SAV clip demos

Prerequisites: Run PACR cell + TST cell before this cell.
Install: pip install gradio nest_asyncio -q
================================================================================
"""


import gradio as gr
import numpy as np
import cv2
import torch
import torch.nn.functional as F
import os, gc, math, tempfile, io, warnings
from collections import defaultdict
from PIL import Image, ImageDraw, ImageFont
from torchvision import transforms
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')

# ============================================================================
# DEVICE & PATHS
# ============================================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PACR_CHECKPOINT = "/kaggle/input/models/YOUR_KAGGLE_USERNAME/pacr-bound/pytorch/default/1/best (1) (1).pth"
TST_CHECKPOINT  = "/kaggle/input/models/YOUR_KAGGLE_USERNAME/tst-small-best/pytorch/default/1/tst_best (1).pth"
SAMPLE_DIR      = "/kaggle/input/datasets/YOUR_KAGGLE_USERNAME/sav-dataset/SAV/data/sav/video_clips"

# ============================================================================
# ACTION METADATA (matches your inference pipeline exactly)
# ============================================================================
ACTION_NAMES = [
    'sit','stand','look_forward','look_sideways','read',
    'flip_books','touch_sth','raise_hand','hands_down','take_notes',
    'applaud','bend','turn_around','talk_with_others','answer_questions'
]
ACTION_DISPLAY = [
    'Sitting','Standing','Looking forward','Looking sideways','Reading',
    'Flipping books','Touching sth','Raising hand','Hands down','Taking notes',
    'Applauding','Bending','Turning around','Talking','Answering',
]
ACTION_SHORT = [
    'Sit','Stand','Fwd','Side','Read',
    'Flip','Touch','Hand+','Hand-','Notes',
    'Clap','Bend','Turn','Talk','Ans',
]
GROUPS = {
    'Posture':   [0,1,11,12],
    'Attention': [2,3,7,8],
    'Activity':  [4,5,6,9],
    'Social':    [10,13,14],
}
GROUP_COLORS = {'Posture':'#5C72F0','Attention':'#14A87A','Activity':'#F5A623','Social':'#E05A7B'}

ENGAGE_WEIGHTS = np.array([
    0.3, 0.0, 0.8, -0.6, 0.9, 0.7, 0.0, 1.0, 0.0, 0.9,
    0.0, -0.2, -0.7, -0.8, 1.0
])
M_TOTAL = np.sum(np.abs(ENGAGE_WEIGHTS))

BOX_PALETTE = [
    ('#5C72F0','#EEF0FE'), ('#14A87A','#E6F9F3'), ('#F5A623','#FEF7E6'),
    ('#E05A7B','#FDE8EF'), ('#8B5CF6','#F0EBFD'), ('#0EA5E9','#E0F5FE'),
    ('#10B981','#D1FAE5'), ('#F97316','#FFF0E6'),
]

# Card colors
BG='#F7F8FC'; CARD_BG='#FFFFFF'; BORDER='#E4E7EF'
TEXT_PRI='#1A1D2E'; TEXT_SEC='#6B7280'; ACCENT='#5C72F0'
GREEN='#14A87A'; AMBER='#F5A623'; RED='#E05A7B'

# ============================================================================
# FONT + DRAWING HELPERS
# ============================================================================
def _font(size, bold=False):
    for tmpl in ['/usr/share/fonts/truetype/dejavu/DejaVuSans{}.ttf',
                 '/usr/share/fonts/truetype/liberation/LiberationSans{}-Regular.ttf']:
        path = tmpl.format('-Bold' if bold else '')
        if os.path.exists(path):
            try: return ImageFont.truetype(path, size)
            except: pass
    return ImageFont.load_default()

def _rrect(draw, xy, radius, fill=None, outline=None, width=1):
    draw.rounded_rectangle(list(xy), radius=radius, fill=fill, outline=outline, width=width)

def _eng_color(s): return GREEN if s>=0.65 else (AMBER if s>=0.45 else RED)
def _eng_label(s): return 'Engaged' if s>=0.65 else ('Moderate' if s>=0.45 else 'Disengaged')
def _eng_level(s): return 'HIGH' if s>=0.65 else ('MEDIUM' if s>=0.45 else 'LOW')

def compute_engagement(probs):
    return float((np.dot(probs, ENGAGE_WEIGHTS) + M_TOTAL) / (2 * M_TOTAL))

# ============================================================================
# STUDENT TRACKER (IoU-based stable IDs)
# ============================================================================
class StudentTracker:
    def __init__(self, iou_thr=0.40, max_missing=5):
        self.iou_thr=iou_thr; self.max_missing=max_missing
        self._tracks={}; self._next_id=1

    def update(self, boxes):
        if boxes is None or len(boxes)==0:
            self._age(); return np.zeros((0,4),dtype=np.float32), np.zeros(0,dtype=int)
        boxes=np.asarray(boxes, dtype=np.float32); N=len(boxes)
        if not self._tracks: return boxes, self._reg(boxes)
        tids=list(self._tracks.keys())
        tboxes=np.stack([self._tracks[t]['box'] for t in tids])
        iou=self._iou(boxes, tboxes)
        ad, at = {}, set()
        for i,j,v in sorted([(i,j,iou[i,j]) for i in range(N) for j in range(len(tids))], key=lambda x:-x[2]):
            if v<self.iou_thr: break
            if i in ad or j in at: continue
            ad[i]=tids[j]; at.add(j)
            self._tracks[tids[j]]['box']=boxes[i]; self._tracks[tids[j]]['missing']=0
        ids=np.zeros(N,dtype=int)
        for i in range(N):
            if i in ad: ids[i]=ad[i]
            else:
                nid=self._next_id; self._next_id+=1
                self._tracks[nid]={'box':boxes[i],'missing':0}; ids[i]=nid
        for j,t in enumerate(tids):
            if j not in at:
                self._tracks[t]['missing']+=1
                if self._tracks[t]['missing']>self.max_missing: del self._tracks[t]
        return boxes, ids

    def _reg(self, boxes):
        ids=[]
        for b in boxes:
            tid=self._next_id; self._next_id+=1
            self._tracks[tid]={'box':b,'missing':0}; ids.append(tid)
        return np.array(ids, dtype=int)
    def _age(self):
        for t in list(self._tracks):
            self._tracks[t]['missing']+=1
            if self._tracks[t]['missing']>self.max_missing: del self._tracks[t]
    @staticmethod
    def _iou(a,b):
        ix1=np.maximum(a[:,0,None],b[None,:,0]); iy1=np.maximum(a[:,1,None],b[None,:,1])
        ix2=np.minimum(a[:,2,None],b[None,:,2]); iy2=np.minimum(a[:,3,None],b[None,:,3])
        inter=np.maximum(0,ix2-ix1)*np.maximum(0,iy2-iy1)
        aa=(a[:,2]-a[:,0])*(a[:,3]-a[:,1]); ab=(b[:,2]-b[:,0])*(b[:,3]-b[:,1])
        return inter/(aa[:,None]+ab[None,:]-inter+1e-6)

# ============================================================================
# MODEL LOADING
# ============================================================================
_normalize = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])

pacr_model = None
tst_model = None

def load_models():
    global pacr_model, tst_model

    # PACR
    pacr_config = PACRConfig()
    pacr_model_obj = PACRDETRDetector(pacr_config)
    ckpt = torch.load(PACR_CHECKPOINT, map_location='cpu', weights_only=False)
    state = ckpt.get('model_state_dict', ckpt)
    state = {k.replace('module.',''): v for k,v in state.items()}
    ms = pacr_model_obj.state_dict()
    for k,v in state.items():
        if k in ms and v.shape==ms[k].shape: ms[k]=v
    pacr_model_obj.load_state_dict(ms)
    pacr_model_obj.to(DEVICE).eval()
    pacr_model = pacr_model_obj
    print("✓ PACR loaded")

    # TST
    tst_config = Config()
    tst_config.USE_GRADIENT_CHECKPOINT = False
    tst_model_obj = TST(tst_config)
    ckpt = torch.load(TST_CHECKPOINT, map_location='cpu', weights_only=False)
    state = ckpt.get('tst_state_dict', ckpt)
    state = {k.replace('module.',''): v for k,v in state.items()}
    tst_model_obj.load_state_dict(state, strict=False)
    tst_model_obj.to(DEVICE).eval()
    tst_model = tst_model_obj
    print("✓ TST loaded")

try:
    load_models()
except Exception as e:
    print(f"⚠ Model loading: {e}\n  Running in demo mode")

# ============================================================================
# DETECTION & CLASSIFICATION
# ============================================================================
def detect_students(bgr_frame):
    if pacr_model is None:
        h,w=bgr_frame.shape[:2]
        return np.array([[0.1,0.1,0.3,0.5],[0.4,0.1,0.6,0.5],[0.65,0.15,0.85,0.55]]), np.array([0.95,0.91,0.88])

    img = cv2.resize(bgr_frame, (640,640))
    img = img[:,:,::-1].astype(np.float32)/255.0
    img = (img-np.array([0.485,0.456,0.406]))/np.array([0.229,0.224,0.225])
    tensor = torch.tensor(np.transpose(img,(2,0,1)), dtype=torch.float32).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        out = pacr_model(tensor)
    obj = out['pred_obj'][0].sigmoid().squeeze(-1).cpu().numpy()
    boxes = out['pred_boxes'][0].cpu().numpy()  # cxcywh norm

    mask = obj > 0.45
    boxes_f = boxes[mask]; scores = obj[mask]
    if len(boxes_f)==0: return np.zeros((0,4)), np.zeros(0)

    # cxcywh → xyxy (normalized)
    cx,cy,bw,bh = boxes_f[:,0],boxes_f[:,1],boxes_f[:,2],boxes_f[:,3]
    xyxy = np.stack([cx-bw/2, cy-bh/2, cx+bw/2, cy+bh/2], axis=1)
    return xyxy, scores

def classify_students(frame_buffer, boxes_norm, max_batch=4):
    if tst_model is None or len(boxes_norm)==0:
        n=max(len(boxes_norm),1)
        p=np.random.rand(n,15)*0.3; p[:,0]=np.random.uniform(0.7,0.95,n)
        p[:,2]=np.random.uniform(0.4,0.8,n); return p[:len(boxes_norm)]

    K, T = 3, 16
    needed = K * T  # 48 frames total

    # Pad or trim buffer to exactly needed frames
    buf = frame_buffer
    if len(buf) < needed:
        buf = buf[0:1] * (needed - len(buf)) + buf
    buf = buf[:needed]  # trim if longer

    all_probs = []
    for i in range(0, len(boxes_norm), max_batch):
        batch_boxes = boxes_norm[i:i+max_batch]
        tubes = []
        for box in batch_boxes:
            crops = []
            for f in buf:  # exactly K*T frames
                _, H, W = f.shape
                x1=max(0,int(box[0]*W)); y1=max(0,int(box[1]*H))
                x2=min(W,int(box[2]*W)); y2=min(H,int(box[3]*H))
                if x2<=x1 or y2<=y1:
                    crop = torch.zeros(3, 224, 224)
                else:
                    crop = f[:, y1:y2, x1:x2]
                    crop = F.interpolate(crop.unsqueeze(0), size=(224,224),
                                        mode='bilinear', align_corners=False).squeeze(0)
                crops.append(crop)
            # crops: list of K*T tensors → stack to (K*T, 3, 224, 224)
            # then reshape to (K, T, 3, 224, 224)
            tube = torch.stack(crops).view(K, T, 3, 224, 224)
            tubes.append(tube)

        # batch_t: (batch, K, T, 3, 224, 224)
        batch_t = torch.stack(tubes).to(DEVICE)
        with torch.no_grad(), torch.cuda.amp.autocast():
            logits = tst_model(batch_t)
            probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        del batch_t; torch.cuda.empty_cache()

    return np.concatenate(all_probs, axis=0)

# ============================================================================
# POLISHED FRAME ANNOTATION (PIL-based from your inference code)
# ============================================================================
def annotate_frame_pil(bgr, boxes_norm, track_ids, student_results, frame_idx):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(rgb)
    draw = ImageDraw.Draw(img, 'RGBA')
    IW,IH = img.size
    f_id=_font(13,bold=True); f_act=_font(11)

    for i, box in enumerate(boxes_norm):
        tid = int(track_ids[i]) if i<len(track_ids) else i+1
        stroke, pill_bg = BOX_PALETTE[(tid-1)%len(BOX_PALETTE)]
        x1,y1,x2,y2 = int(box[0]*IW),int(box[1]*IH),int(box[2]*IW),int(box[3]*IH)

        # box
        draw.rounded_rectangle([x1,y1,x2,y2], radius=6, fill=stroke+'22', outline=stroke, width=2)

        # ID badge
        badge=f'S{tid}'
        bb=draw.textbbox((0,0),badge,font=f_id)
        bw=bb[2]-bb[0]+14; bh=bb[3]-bb[1]+8
        _rrect(draw,[x1,max(0,y1-bh-2),x1+bw,y1],radius=5,fill=stroke)
        draw.text((x1+7,max(bh//2,y1-bh//2-2)),badge,font=f_id,fill='#FFFFFF',anchor='lm')

        # action pills
        if i<len(student_results):
            res=student_results[i]
            py=y2+4
            top_idx=np.argsort(res)[::-1]
            shown=0
            for idx in top_idx:
                if res[idx]<0.40 or shown>=3: break
                txt=f'{ACTION_DISPLAY[idx]}  {int(res[idx]*100)}%'
                bb=draw.textbbox((0,0),txt,font=f_act)
                pw=bb[2]-bb[0]+16; ph=bb[3]-bb[1]+8
                _rrect(draw,[x1,py,x1+pw,py+ph],radius=5,fill=pill_bg,outline=stroke,width=1)
                draw.text((x1+8,py+ph//2),txt,font=f_act,fill=stroke,anchor='lm')
                py+=ph+4; shown+=1

    # HUD
    draw.rectangle([0,0,IW,36],fill='#000000BB')
    draw.text((12,18),f'Frame {frame_idx:05d}',font=_font(12),fill='#FFFFFF',anchor='lm')
    draw.text((IW-12,18),f'{len(boxes_norm)} students',font=_font(12),fill='#CCCCCC',anchor='rm')

    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

# ============================================================================
# STUDENT REPORT CARD (from your inference code)
# ============================================================================
def generate_student_card(tid, probs, student_img=None, stroke_color=None):
    W,H=700,400
    img=Image.new('RGB',(W,H),BG)
    draw=ImageDraw.Draw(img)
    accent=stroke_color or ACCENT

    _rrect(draw,[14,14,W-14,H-14],radius=18,fill=CARD_BG,outline=BORDER,width=1)

    # LEFT PANEL
    PW=178
    _rrect(draw,[26,26,26+PW,H-26],radius=14,fill=BG)
    PH_W,PH_H=130,160; px=26+(PW-PH_W)//2; py=38

    if student_img is not None:
        thumb=student_img.copy().convert('RGB')
        ratio=max(PH_W/max(thumb.width,1),PH_H/max(thumb.height,1))
        thumb=thumb.resize((max(1,int(thumb.width*ratio)),max(1,int(thumb.height*ratio))),Image.LANCZOS)
        left=(thumb.width-PH_W)//2; top=(thumb.height-PH_H)//2
        thumb=thumb.crop((left,top,left+PH_W,top+PH_H))
        mask=Image.new('L',(PH_W,PH_H),0)
        ImageDraw.Draw(mask).rounded_rectangle([0,0,PH_W,PH_H],radius=10,fill=255)
        thumb_rgba=thumb.convert('RGBA'); thumb_rgba.putalpha(mask)
        img.paste(thumb_rgba,(px,py),thumb_rgba)
        _rrect(draw,[px-2,py-2,px+PH_W+2,py+PH_H+2],radius=11,outline=accent,width=2)
    else:
        _rrect(draw,[px,py,px+PH_W,py+PH_H],radius=10,fill=accent)
        draw.text((px+PH_W//2,py+PH_H//2),f'S{tid}',font=_font(44,bold=True),fill='#FFFFFF',anchor='mm')

    draw.text((26+PW//2,py+PH_H+14),f'Student {tid}',font=_font(13,bold=True),fill=TEXT_PRI,anchor='mm')

    # Engagement ring
    es=compute_engagement(probs)
    ec=_eng_color(es); el=_eng_label(es)
    rcx=26+PW//2; rcy=py+PH_H+54; rr=34
    draw.arc([rcx-rr,rcy-rr,rcx+rr,rcy+rr],0,360,fill=BORDER,width=7)
    draw.arc([rcx-rr,rcy-rr,rcx+rr,rcy+rr],-90,-90+max(1,int(es*360)),fill=ec,width=7)
    pct=f'{int(es*100)}%'
    bb=draw.textbbox((0,0),pct,font=_font(15,bold=True))
    draw.text((rcx-(bb[2]-bb[0])//2,rcy-11),pct,font=_font(15,bold=True),fill=ec)
    draw.text((rcx,rcy+12),el,font=_font(9),fill=ec,anchor='mm')

    # RIGHT PANEL
    RX=26+PW+18; RW=W-RX-26
    # Group bars
    draw.text((RX,34),'Behaviour breakdown',font=_font(11,bold=True),fill=TEXT_PRI)
    gy=54
    for gname,gidxs in GROUPS.items():
        gs=float(np.mean(probs[gidxs])); gc=GROUP_COLORS[gname]
        bf=RW-92; bfill=max(0,int(gs*bf))
        draw.text((RX,gy),gname,font=_font(10),fill=TEXT_SEC)
        tx=RX+82
        _rrect(draw,[tx,gy+2,tx+bf,gy+11],radius=4,fill=BORDER)
        if bfill>0: _rrect(draw,[tx,gy+2,tx+bfill,gy+11],radius=4,fill=gc)
        draw.text((tx+bf+5,gy+2),f'{int(gs*100)}%',font=_font(9),fill=gc)
        gy+=22

    draw.line([(RX,gy+4),(W-30,gy+4)],fill=BORDER,width=1); gy+=14

    # Top action tags
    draw.text((RX,gy),'Top detected actions',font=_font(11,bold=True),fill=TEXT_PRI); gy+=18
    top_idx=np.argsort(probs)[::-1][:8]
    tx,ty=RX,gy
    for idx in top_idx:
        p=probs[idx]
        if p<0.22: break
        name=ACTION_DISPLAY[idx]
        bb=draw.textbbox((0,0),name,font=_font(11))
        tw=bb[2]-bb[0]+18
        if tx+tw>W-30: tx=RX; ty+=27
        if p>=0.65: bg_c,txt_c='#E6F9F3','#0D7A57'
        elif p>=0.40: bg_c,txt_c='#FEF3E2','#B87100'
        else: bg_c,txt_c='#F1F3FE','#3A50C4'
        _rrect(draw,[tx,ty,tx+tw,ty+22],radius=5,fill=bg_c)
        draw.text((tx+9,ty+11),name,font=_font(11),fill=txt_c,anchor='lm')
        tx+=tw+6
    gy=ty+36

    draw.line([(RX,gy),(W-30,gy)],fill=BORDER,width=1); gy+=12

    # Mini bars
    draw.text((RX,gy),'All actions',font=_font(11,bold=True),fill=TEXT_PRI); gy+=14
    col_w=RW//3
    for i,(sn,p) in enumerate(zip(ACTION_SHORT,probs)):
        cx=RX+(i%3)*col_w; cy=gy+(i//3)*17
        draw.text((cx,cy),sn,font=_font(9),fill=TEXT_SEC)
        bx=cx+44; bf=col_w-52
        _rrect(draw,[bx,cy+2,bx+bf,cy+9],radius=3,fill=BORDER)
        fw=max(2,int(p*bf))
        fc=GREEN if p>=0.5 else (AMBER if p>=0.3 else BORDER)
        _rrect(draw,[bx,cy+2,bx+fw,cy+9],radius=3,fill=fc)

    return img

# ============================================================================
# CLASS SUMMARY DASHBOARD (matplotlib)
# ============================================================================
def generate_class_summary(all_probs, video_id=''):
    N=all_probs.shape[0]
    fig=plt.figure(figsize=(13,8.5),facecolor=BG,dpi=120)

    gs=fig.add_gridspec(2,3,hspace=0.44,wspace=0.32,left=0.06,right=0.97,top=0.88,bottom=0.07)
    ax_r=fig.add_subplot(gs[0,0],polar=True)
    ax_b=fig.add_subplot(gs[0,1:])
    ax_h=fig.add_subplot(gs[1,:2])
    ax_e=fig.add_subplot(gs[1,2])

    fig.text(0.5,0.95,'Classroom Engagement Report',ha='center',fontsize=16,fontweight='bold',color=TEXT_PRI)
    sub=f'{N} students detected'
    if video_id: sub=f'{video_id} — {sub}'
    fig.text(0.5,0.915,sub,ha='center',fontsize=10,color=TEXT_SEC)

    # Radar
    gs_scores={g:float(np.mean(all_probs[:,idx])) for g,idx in GROUPS.items()}
    labels=list(gs_scores.keys()); vals=list(gs_scores.values())+[list(gs_scores.values())[0]]
    angles=[n/len(labels)*2*math.pi for n in range(len(labels))]+[0]
    ax_r.set_facecolor(CARD_BG); ax_r.plot(angles,vals,'o-',lw=2,color=ACCENT)
    ax_r.fill(angles,vals,alpha=0.18,color=ACCENT)
    ax_r.set_xticks(angles[:-1]); ax_r.set_xticklabels(labels,fontsize=9,color=TEXT_PRI)
    ax_r.tick_params(axis='x',pad=18); ax_r.set_ylim(0,1)
    ax_r.set_title('Group breakdown',fontsize=11,color=TEXT_PRI,pad=12)
    ax_r.grid(color=BORDER,lw=0.8)

    # Bars
    mp=all_probs.mean(0); si=np.argsort(mp)[::-1]
    clrs=[GREEN if mp[i]>=0.6 else (AMBER if mp[i]>=0.3 else '#D0D5E8') for i in si]
    ax_b.set_facecolor(CARD_BG)
    bars=ax_b.barh(range(15),mp[si],color=clrs,height=0.65,zorder=2)
    ax_b.set_yticks(range(15)); ax_b.set_yticklabels([ACTION_DISPLAY[i] for i in si],fontsize=9)
    ax_b.set_xlim(0,1.08); ax_b.set_title('Action frequencies',fontsize=11,color=TEXT_PRI)
    ax_b.invert_yaxis(); ax_b.spines[['top','right']].set_visible(False)
    for bar,v in zip(bars,mp[si]): ax_b.text(v+0.01,bar.get_y()+bar.get_height()/2,f'{v:.0%}',va='center',fontsize=8,color=TEXT_SEC)

    # Heatmap
    ax_h.set_facecolor(CARD_BG)
    im=ax_h.imshow(all_probs,aspect='auto',cmap='YlGn',vmin=0,vmax=1)
    ax_h.set_xticks(range(15)); ax_h.set_xticklabels(ACTION_SHORT,fontsize=8,rotation=35,ha='right')
    ax_h.set_yticks(range(N)); ax_h.set_yticklabels([f'S{i+1}' for i in range(N)],fontsize=8)
    ax_h.set_title('Student × action heatmap',fontsize=11,color=TEXT_PRI)
    fig.colorbar(im,ax=ax_h,shrink=0.85,pad=0.02)

    # Donut
    scores=np.array([compute_engagement(p) for p in all_probs])
    eng=int((scores>=0.65).sum()); mod=int(((scores>=0.45)&(scores<0.65)).sum()); dis=int((scores<0.45).sum())
    ax_e.set_facecolor(CARD_BG)
    _,_,at=ax_e.pie([eng,mod,dis],colors=[GREEN,AMBER,RED],autopct='%1.0f%%',startangle=90,pctdistance=0.78,
                     wedgeprops=dict(width=0.48,edgecolor='white',linewidth=2))
    for a in at: a.set_fontsize(9); a.set_color('white'); a.set_fontweight('bold')
    ax_e.set_title('Engagement',fontsize=11,color=TEXT_PRI)
    patches=[mpatches.Patch(color=c,label=f'{l} ({s})') for c,l,s in zip([GREEN,AMBER,RED],['Engaged','Moderate','Disengaged'],[eng,mod,dis])]
    ax_e.legend(handles=patches,fontsize=8,loc='lower center',bbox_to_anchor=(0.5,-0.22),frameon=False)
    ax_e.text(0,0,f'{N}\nstudents',ha='center',va='center',fontsize=9,color=TEXT_PRI,linespacing=1.4)

    buf=io.BytesIO(); fig.savefig(buf,dpi=120,bbox_inches='tight',facecolor=BG); plt.close(fig)
    buf.seek(0); return Image.open(buf).convert('RGB')

# ============================================================================
# MAIN ANALYSIS FUNCTION
# ============================================================================
def analyze_video(video_path, progress=gr.Progress(track_tqdm=True)):
    if video_path is None: return None, None, None, [], "⚠ Please upload a video."

    progress(0, desc="Opening video...")
    cap=cv2.VideoCapture(video_path)
    fps=cap.get(cv2.CAP_PROP_FPS) or 24; w=int(cap.get(3)); h=int(cap.get(4))
    total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    vid=os.path.splitext(os.path.basename(video_path))[0]

    frames_bgr=[]; 
    while True:
        ret,f=cap.read()
        if not ret: break
        frames_bgr.append(f)
    cap.release()
    if not frames_bgr: return None,None,None,[],"⚠ Could not read video."

    progress(0.15, desc="Detecting students...")
    tracker=StudentTracker()
    # Build frame buffer (normalized tensors)
    frame_buffer=[]
    for f in frames_bgr:
        rgb=cv2.cvtColor(f,cv2.COLOR_BGR2RGB)
        t=torch.from_numpy(rgb).permute(2,0,1).float()/255.0
        frame_buffer.append(_normalize(t))

    # Detect on middle frame
    mid_idx=len(frames_bgr)//2
    boxes_norm, scores = detect_students(frames_bgr[mid_idx])
    if len(boxes_norm)==0: return None,None,None,[],"⚠ No students detected."
    boxes_norm, track_ids = tracker.update(boxes_norm)

    progress(0.4, desc=f"Classifying {len(boxes_norm)} students...")
    all_probs = classify_students(frame_buffer, boxes_norm)

    progress(0.6, desc="Rendering annotated video...")
    out_path=tempfile.mktemp(suffix='.mp4')
    fourcc=cv2.VideoWriter_fourcc(*'mp4v')
    writer=cv2.VideoWriter(out_path, fourcc, fps, (w,h))
    for fi, bgr in enumerate(frames_bgr):
        ann = annotate_frame_pil(bgr, boxes_norm, track_ids, all_probs, fi+1)
        writer.write(ann)
    writer.release()

    progress(0.75, desc="Generating report cards...")
    # Crop student images
    last = frames_bgr[-1]
    cards=[]
    for i in range(len(boxes_norm)):
        box=boxes_norm[i]; H,W=last.shape[:2]
        x1,y1=max(0,int(box[0]*W)),max(0,int(box[1]*H))
        x2,y2=min(W,int(box[2]*W)),min(H,int(box[3]*H))
        crop=last[y1:y2,x1:x2]
        crop_pil=Image.fromarray(cv2.cvtColor(crop,cv2.COLOR_BGR2RGB)) if crop.size>0 else None
        tid=int(track_ids[i]) if i<len(track_ids) else i+1
        stroke,_=BOX_PALETTE[(tid-1)%len(BOX_PALETTE)]
        card=generate_student_card(tid, all_probs[i], crop_pil, stroke)
        cards.append(card)

    progress(0.9, desc="Building dashboard...")
    # Annotated keyframe
    ann_key = annotate_frame_pil(frames_bgr[mid_idx], boxes_norm, track_ids, all_probs, mid_idx+1)
    ann_key_rgb = cv2.cvtColor(ann_key, cv2.COLOR_BGR2RGB)

    # Summary dashboard
    summary_img = generate_class_summary(all_probs, vid)

    # Text summary
    summary = f"## ✅ Analysis Complete\n\n"
    summary += f"**Video:** {vid} — {total} frames, {fps:.0f} FPS, {w}×{h}\n\n"
    summary += f"**Students detected:** {len(boxes_norm)}\n\n"
    summary += "| Student | Engagement | Level | Top Actions |\n|---|---|---|---|\n"
    for i in range(len(all_probs)):
        tid=int(track_ids[i]) if i<len(track_ids) else i+1
        es=compute_engagement(all_probs[i])
        lv=_eng_level(es)
        top3=np.argsort(all_probs[i])[::-1][:3]
        acts=", ".join([f"{ACTION_DISPLAY[j]} ({all_probs[i][j]:.0%})" for j in top3 if all_probs[i][j]>0.25])
        summary+=f"| S{tid} | {es:.0%} | {lv} | {acts} |\n"
    avg=np.mean([compute_engagement(p) for p in all_probs])
    summary+=f"\n**Class Average: {avg:.0%} ({_eng_level(avg)})**"

    progress(1.0)
    return out_path, ann_key_rgb, summary_img, cards, summary

# ============================================================================
# SINGLE FRAME DETECTION
# ============================================================================
def detect_frame(image):
    if image is None: return None, "⚠ Upload an image."
    bgr=cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    boxes,scores=detect_students(bgr)
    if len(boxes)==0: return np.array(image),"No students detected."

    tracker=StudentTracker()
    boxes, tids = tracker.update(boxes)
    ann=annotate_frame_pil(bgr, boxes, tids, [], 1)
    ann_rgb=cv2.cvtColor(ann, cv2.COLOR_BGR2RGB)

    info=f"**Detected {len(boxes)} students**\n\n"
    for i,(box,sc) in enumerate(zip(boxes,scores)):
        tid=int(tids[i])
        info+=f"- S{tid}: conf={sc:.2f}, box=[{box[0]:.3f}, {box[1]:.3f}, {box[2]:.3f}, {box[3]:.3f}]\n"
    return ann_rgb, info

# ============================================================================
# SAMPLE CLIPS
# ============================================================================
def get_samples():
    if os.path.exists(SAMPLE_DIR):
        return sorted([f for f in os.listdir(SAMPLE_DIR) if f.endswith('.mp4')])[:15]
    return []

def run_sample(clip_name, progress=gr.Progress(track_tqdm=True)):
    path=os.path.join(SAMPLE_DIR, clip_name)
    if not os.path.exists(path): return None,None,None,[],"Clip not found."
    return analyze_video(path, progress)

# ============================================================================
# GRADIO APP
# ============================================================================
THEME = gr.themes.Soft(
    primary_hue="teal",
    secondary_hue="slate",
    neutral_hue="slate",
)
CSS = """
.main-header { text-align:center; padding:10px 0 5px; }
.main-header h1 { color:#0E7490; font-size:26px; margin:0; }
.main-header p { color:#64748B; font-size:13px; margin:2px 0; }
footer { display:none !important; }
.card-gallery { border-radius:12px; }
"""

with gr.Blocks(theme=THEME, css=CSS, title="Classroom Behavior Recognition") as app:

    gr.HTML("""
    <div class="main-header">
        <h1>🎓 Classroom Student Behavior Recognition</h1>
        <p>PACR-DETR Detection (89.14% AP@50) + TST Classification (71.49% E2E mAP) · 15 Multi-Label Actions · SAV Dataset</p>
        <p style="font-size:11px; color:#94A3B8;">Srinithi R · Divya Maki S · Thirumoorthy K · Mepco Schlenk Engineering College</p>
    </div>
    """)

    with gr.Tabs():

        # ─── TAB 1: FULL VIDEO ANALYSIS ───
        with gr.TabItem("🎬 Analyze Video"):
            gr.Markdown("Upload a classroom video clip. The system detects students, classifies 15 actions, and generates engagement analytics.")
            with gr.Row():
                with gr.Column(scale=1):
                    vid_in = gr.Video(label="Upload Classroom Video")
                    vid_btn = gr.Button("🚀 Run Full Analysis", variant="primary", size="lg")
                with gr.Column(scale=1):
                    vid_out = gr.Video(label="Annotated Output")

            with gr.Row():
                with gr.Column(scale=1):
                    key_frame = gr.Image(label="Annotated Keyframe")
                with gr.Column(scale=1):
                    dash_img = gr.Image(label="Class Engagement Dashboard")

            gr.Markdown("### 📋 Per-Student Engagement Report Cards")
            card_gallery = gr.Gallery(label="Student Cards", columns=3, height=450, object_fit="contain")

            vid_summary = gr.Markdown(label="Summary")

            vid_btn.click(analyze_video, inputs=vid_in,
                         outputs=[vid_out, key_frame, dash_img, card_gallery, vid_summary])

        # ─── TAB 2: SINGLE FRAME ───
        with gr.TabItem("📸 Detect Students"):
            gr.Markdown("Upload a single classroom image for PACR-DETR student detection.")
            with gr.Row():
                with gr.Column():
                    img_in = gr.Image(label="Upload Frame", type="pil")
                    img_btn = gr.Button("🔍 Detect", variant="primary", size="lg")
                with gr.Column():
                    img_out = gr.Image(label="Detection Result")
                    img_info = gr.Markdown()
            img_btn.click(detect_frame, inputs=img_in, outputs=[img_out, img_info])

        # ─── TAB 3: SAMPLE DEMOS ───
        with gr.TabItem("🎮 SAV Demos"):
            gr.Markdown("Try the system on pre-loaded SAV dataset clips.")
            with gr.Row():
                with gr.Column(scale=1):
                    samples = get_samples()
                    sample_dd = gr.Dropdown(choices=samples, label="Select clip",
                                           value=samples[0] if samples else None)
                    sample_btn = gr.Button("▶️ Run", variant="primary", size="lg")
                with gr.Column(scale=1):
                    s_vid = gr.Video(label="Annotated Output")

            with gr.Row():
                s_key = gr.Image(label="Keyframe")
                s_dash = gr.Image(label="Dashboard")

            s_cards = gr.Gallery(label="Student Cards", columns=3, height=400, object_fit="contain")
            s_summary = gr.Markdown()

            sample_btn.click(run_sample, inputs=sample_dd,
                            outputs=[s_vid, s_key, s_dash, s_cards, s_summary])

    gr.HTML("""
    <div style="text-align:center; padding:12px; margin-top:16px; border-top:1px solid #e0e0e0; color:#94A3B8; font-size:11px;">
        Spatiotemporal Framework for Automated Multi-Label Student Behavior Recognition in Classrooms<br>
        E = (ΣPᵢWᵢ + M) / 2M  |  M = Σ|Wᵢ| = 7.6  |  HIGH > 65%  |  MEDIUM 45–65%  |  LOW < 45%<br>
        © 2025 Mepco Schlenk Engineering College, Sivakasi
    </div>
    """)

# ============================================================================
# LAUNCH
# ============================================================================
gc.collect(); torch.cuda.empty_cache()
app.launch(server_name="0.0.0.0", share=True, debug=False, show_error=True)

âœ“ Loaded pretrained ResNet50 weights
✓ PACR loaded
✓ TST loaded
* Running on local URL:  http://0.0.0.0:7861
* Running on public URL: https://236b43fcea097b605a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
